In [ ]:
from typing import List, Dict, Optional
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import json
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_linear_schedule_with_warmup
from scipy.stats import pearsonr
import spacy
import pandas as pd
import re
from tqdm import tqdm
import os
from datetime import date
from datetime import datetime
from sklearn.metrics import f1_score

In [ ]:
import transformers
import sklearn
print(torch.__version__)
print(transformers.__version__)
print(sklearn.__version__)
print(spacy.__version__)

In [ ]:
cuda_available = torch.cuda.is_available()
print(f"CUDA Available: {cuda_available}")
if cuda_available:
    print(f"Number of GPUs available: {torch.cuda.device_count()}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")

In [ ]:
import pickle
from tqdm import tqdm

def save_pkl(tgt_list, svg_path):
    with open(svg_path, "wb") as f:
        pickle.dump(tgt_list, f)

def load_pkl(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data

# 0. Dataset Creation

In [ ]:
from dataclasses import dataclass, field
from typing import List, Optional

@dataclass
class SRLFrame:
    predicate_word_idx: int
    labels: List[str]
    predicate_form: Optional[str] = None
    arg_head_idx: Optional[List[int]] = None

@dataclass
class UtteranceSample:
    words: List[str]
    frames: List[SRLFrame]
    formality: Optional[float] = None

def create_utterance_samples(data_path):
    samples = []
    with open(data_path, 'r', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line)
            samples.append(UtteranceSample(
                words=data["words"],
                frames=[SRLFrame(**fr) for fr in data["frames"]],
                formality=data.get("formality", None)
            ))
    return samples    
    

def create_formality_samples_from_df(df) -> List[UtteranceSample]:
    samples = []

    for _, row in df.iterrows():
        text = row["sent"]
        label = row["label"]

        if not isinstance(text, str) or not text.strip():
            continue

        samples.append(
            UtteranceSample(
                words=text.strip().split(),
                frames=[],
                formality=float(label)
            )
        )

    return samples

# 1. SRL Dataset (Utterance Level)

In [ ]:
class SRLDataset(Dataset):
    """
    One item = one utterance with MULTIPLE SRL frames.
    BERT input: [CLS] <utterance> [SEP] (single segment).
    Per-frame tensors are returned as a list of dicts.
    Utterance-level target: formality score.
    """
    def __init__(
        self,
        samples: List[UtteranceSample],
        tokenizer,
        label2id: Dict[str, int],
        max_length: int = 256,
        debug_print: bool = False
    ):
        self.samples = samples
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.id2label = {v: k for k, v in label2id.items()}
        self.max_length = max_length
        self.debug_print = debug_print

    def __len__(self):
        return len(self.samples)

    def _tokenize_utterance(self, words: List[str]):
        enc = self.tokenizer(
            words,
            is_split_into_words=True,
            add_special_tokens=False,
            return_attention_mask=False,
            return_token_type_ids=False,
            truncation=True,
            max_length=self.max_length
        )
        return enc

    # def _build_word_first_wp_fullidx(self, words: List[str]) -> List[int]:
    #     tmp = self.tokenizer(words, is_split_into_words=True, return_offsets_mapping=False, truncation=True, max_length=self.max_length)
    #     word_ids = tmp.word_ids()
    #     first_pos_by_wid = {}
    #     for pos, wid in enumerate(word_ids):
    #         if wid is not None and wid not in first_pos_by_wid:
    #             first_pos_by_wid[wid] = pos
    #     n_tokenized = max(first_pos_by_wid.keys()) + 1 if first_pos_by_wid else 0
    #     return [first_pos_by_wid[w] for w in range(min(len(words), n_tokenized))]

    def _build_word_first_wp_fullidx(self, words):
        tmp = self.tokenizer(
            words,
            is_split_into_words=True,
            add_special_tokens=False,
            return_attention_mask=False,
            return_token_type_ids=False,
            truncation=False,
        )

        first_pos_by_wid = {}
        for pos, wid in enumerate(tmp.word_ids()):
            if wid is not None and wid not in first_pos_by_wid:
                first_pos_by_wid[wid] = pos

        # +1 if you later prepend [CLS] manually
        return [first_pos_by_wid[w] + 1 if w in first_pos_by_wid else -1 for w in range(len(words))]

    def _normalize_srl_label(self, lbl: str) -> str:
        return lbl.replace("R-", "").replace("C-", "")
    
    #updated in Apr 10
    def _frame_to_tensors(self, kept_n_words: int, frame) -> Optional[Dict[str, torch.Tensor]]:
        # If the predicate was truncated away, skip this frame
        if frame.predicate_word_idx is None or frame.predicate_word_idx >= kept_n_words:
            return None

        norm_labels = [self._normalize_srl_label(lbl) for lbl in frame.labels[:kept_n_words]]
        unk_id = self.label2id.get("O", 0)
        label_ids = [self.label2id.get(lbl, unk_id) for lbl in norm_labels]

        # If labels are still somehow shorter than kept_n_words, pad with O
        if len(label_ids) < kept_n_words:
            pad_n = kept_n_words - len(label_ids)
            label_ids += [unk_id] * pad_n
            norm_labels += ["O"] * pad_n

        role_ids = [0] * kept_n_words
        for i, tag in enumerate(norm_labels):
            if i == frame.predicate_word_idx:
                role_ids[i] = 1
            if "ARG0" in tag:
                role_ids[i] = 2
            elif "ARG1" in tag:
                role_ids[i] = 3
            elif "ARG2" in tag:
                role_ids[i] = 4
            elif "ARGM" in tag:
                role_ids[i] = 5

        arg0_mask = [1 if "ARG0" in tag else 0 for tag in norm_labels]
        arg1_mask = [1 if "ARG1" in tag else 0 for tag in norm_labels]
        arg2_mask = [1 if "ARG2" in tag else 0 for tag in norm_labels]
        argm_mask = [1 if "ARGM" in tag else 0 for tag in norm_labels]

        out = {
            "labels":        torch.tensor(label_ids, dtype=torch.long),
            "pred_word_idx": torch.tensor(frame.predicate_word_idx, dtype=torch.long),
            "role_ids":      torch.tensor(role_ids, dtype=torch.long),
            "arg0_mask":     torch.tensor(arg0_mask, dtype=torch.long),
            "arg1_mask":     torch.tensor(arg1_mask, dtype=torch.long),
            "arg2_mask":     torch.tensor(arg2_mask, dtype=torch.long),
            "argm_mask":     torch.tensor(argm_mask, dtype=torch.long),
        }

        if getattr(frame, "arg_head_idx", None) is not None:
            clipped = []
            for x in frame.arg_head_idx[:5]:
                if isinstance(x, int) and x >= kept_n_words:
                    clipped.append(-1)
                else:
                    clipped.append(x)
            out["arg_head_idx"] = torch.tensor(clipped, dtype=torch.long)

        return out


    def __getitem__(self, idx):
        instance = self.samples[idx]
        words    = instance.words

        enc_utt      = self._tokenize_utterance(words)
        sent_wp_ids  = enc_utt["input_ids"]
        input_ids    = [self.tokenizer.cls_token_id] + sent_wp_ids + [self.tokenizer.sep_token_id]
        ttids        = [0] * len(input_ids)
        attn_mask    = [1] * len(input_ids)

        word_first_wp_fullidx = self._build_word_first_wp_fullidx(words)
        n_words  = len(word_first_wp_fullidx)

        if len(input_ids) > self.max_length:
            input_ids = input_ids[:self.max_length]
            ttids     = ttids[:self.max_length]
            attn_mask = attn_mask[:self.max_length]
            max_pos   = self.max_length - 1
            word_first_wp_fullidx = [min(p, max_pos) for p in word_first_wp_fullidx]

        if self.debug_print:
            toks = self.tokenizer.convert_ids_to_tokens(input_ids, skip_special_tokens=False)
            print(f"[DeBug idx={idx}] " + " ".join(toks))
        
        #Apr 10 updated
        frame_dicts = []
        for fr in instance.frames:
            if fr is None:
                continue
            if getattr(fr, "labels", None) is None or getattr(fr, "predicate_word_idx", None) is None:
                continue
            frame_t = self._frame_to_tensors(n_words, fr)
            if frame_t is not None:
                frame_dicts.append(frame_t)

        res = {
            "input_ids":             torch.tensor(input_ids, dtype=torch.long),
            "token_type_ids":        torch.tensor(ttids,     dtype=torch.long),
            "attention_mask":        torch.tensor(attn_mask, dtype=torch.long),
            "word_first_wp_fullidx": torch.tensor(word_first_wp_fullidx, dtype=torch.long),
            "sent_len":              torch.tensor(n_words, dtype=torch.long),
            "frames":                frame_dicts,
        }
        if instance.formality is not None:
            res["formality"] = torch.tensor(instance.formality, dtype=torch.float)
        return res

# 2. Collate Functions

In [ ]:
def srl_collate_ulevel(batch: List[Dict], pad_token_id: int, pad_label_id: int = -100):
    """
    Pads BERT inputs to max_L, word-level tensors to max_n,
    and SRL frames to max_F.
    frames_arg1_mask is the key directional target for the formality model.
    """
    B = len(batch)

    # 1) BERT-level
    max_L          = max(item["input_ids"].size(0) for item in batch)
    input_ids      = torch.full((B, max_L), pad_token_id, dtype=torch.long)
    token_type_ids = torch.zeros((B, max_L), dtype=torch.long)
    attention_mask = torch.zeros((B, max_L), dtype=torch.long)

    # 2) Word-level
    max_n                 = max(int(item["sent_len"]) for item in batch)
    word_first_wp_fullidx = torch.full((B, max_n), -1, dtype=torch.long)
    sent_lens             = torch.zeros((B,), dtype=torch.long)
    sentence_mask         = torch.zeros((B, max_n), dtype=torch.bool)

    # 3) Frames
    max_F = max(len(item.get("frames", [])) for item in batch)
    if max_F == 0:
        max_F = 1

    frames_mask          = torch.zeros((B, max_F), dtype=torch.bool)
    frames_labels        = torch.full((B, max_F, max_n), pad_label_id, dtype=torch.long)
    frames_pred_word_idx = torch.zeros((B, max_F), dtype=torch.long)
    frames_role_ids      = torch.zeros((B, max_F, max_n), dtype=torch.long)
    frames_arg0_mask     = torch.zeros((B, max_F, max_n), dtype=torch.long)
    frames_arg1_mask     = torch.zeros((B, max_F, max_n), dtype=torch.long)
    frames_arg2_mask     = torch.zeros((B, max_F, max_n), dtype=torch.long)
    frames_argm_mask     = torch.zeros((B, max_F, max_n), dtype=torch.long)

    has_arg_head = any(
        any("arg_head_idx" in fr for fr in item.get("frames", []))
        for item in batch
    )
    if has_arg_head:
        max_H = max(
            int(fr["arg_head_idx"].numel())
            for item in batch for fr in item.get("frames", []) if "arg_head_idx" in fr
        )
        frames_arg_head_idx = torch.full((B, max_F, max_H), -1, dtype=torch.long)

    # 4) Utterance-level target
    has_formality = "formality" in batch[0]
    if has_formality:
        formality = torch.zeros((B,), dtype=torch.float)

    for i, item in enumerate(batch):
        L = item["input_ids"].size(0)
        input_ids[i, :L]      = item["input_ids"]
        token_type_ids[i, :L] = item["token_type_ids"]
        attention_mask[i, :L] = item["attention_mask"]

        n = int(item["sent_len"])
        word_first_wp_fullidx[i, :n] = item["word_first_wp_fullidx"]
        sent_lens[i]     = n
        sentence_mask[i, :n] = True

        if has_formality:
            formality[i] = item["formality"]

        for f, fr in enumerate(item.get("frames", [])):
            frames_mask[i, f]             = True
            frames_labels[i, f, :n]       = fr["labels"]
            frames_pred_word_idx[i, f]    = fr["pred_word_idx"]
            frames_role_ids[i, f, :n]     = fr["role_ids"]
            frames_arg0_mask[i, f, :n]    = fr["arg0_mask"]
            frames_arg1_mask[i, f, :n]    = fr["arg1_mask"]
            frames_arg2_mask[i, f, :n]    = fr["arg2_mask"]
            frames_argm_mask[i, f, :n]    = fr["argm_mask"]
            if has_arg_head and "arg_head_idx" in fr:
                h = int(fr["arg_head_idx"].numel())
                frames_arg_head_idx[i, f, :h] = fr["arg_head_idx"]

    res = {
        "input_ids":             input_ids,
        "token_type_ids":        token_type_ids,
        "attention_mask":        attention_mask,
        "word_first_wp_fullidx": word_first_wp_fullidx,
        "sentence_mask":         sentence_mask,
        "sent_lens":             sent_lens,
        "frames_mask":           frames_mask,
        "frames_labels":         frames_labels,
        "frames_pred_word_idx":  frames_pred_word_idx,
        "frames_role_ids":       frames_role_ids,
        "frames_arg0_mask":      frames_arg0_mask,
        "frames_arg1_mask":      frames_arg1_mask,
        "frames_arg2_mask":      frames_arg2_mask,
        "frames_argm_mask":      frames_argm_mask,
    }
    if has_formality:
        res["formality"] = formality
    if has_arg_head:
        res["frames_arg_head_idx"] = frames_arg_head_idx
    return res

In [ ]:
def data_processing_for_loader(
    train_dev: List[UtteranceSample],
    train_sample: List[UtteranceSample],
    dev_sample:   List[UtteranceSample],
    test_sample:  List[UtteranceSample],
    tokenizer
):
    """Build label2id from all frame labels in train+dev, then wrap in SRLDataset."""
    label2id: Dict[str, int] = {}
    for s in train_dev:
        for fr in s.frames:
            if fr is None or fr.labels is None:
                continue
            for l in fr.labels:
                if l not in label2id:
                    label2id[l] = len(label2id)
    id2label = {v: k for k, v in label2id.items()}

    train_bf_loader = SRLDataset(train_sample, tokenizer, label2id, max_length=128, debug_print=False)
    dev_bf_loader   = SRLDataset(dev_sample,   tokenizer, label2id, max_length=128, debug_print=False)
    test_bf_loader  = SRLDataset(test_sample,  tokenizer, label2id, max_length=128, debug_print=False)
    return train_bf_loader, dev_bf_loader, test_bf_loader, label2id, id2label

# 3. SRL-Aware (Directional) Formality Models

### ARG0 -> ARG1 fusion with MLP

In [ ]:
mlp_layer = 300

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoConfig, AutoModel
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

#Fusion with MLP layer
class DirectionalSRL_fusionmlp(nn.Module):
    """
    SRL-aware BERT model for formality prediction with directional SRL + gated fusion.

    Directional design:
      - Query primarily from ARG0, fallback to predicate
      - Attend to ARG1 only if present, otherwise full sentence
      - Add global utterance representation
      - Learn frame importance instead of uniform averaging

    Fusion design:
      - Build a directional SRL representation from [query_source ; context]
      - Build a global text representation from the utterance
      - Use gated residual fusion so SRL selectively updates the global text representation
    """

    def __init__(
        self,
        bert_name: str,
        num_labels: int,
        use_indicator: bool = True,
        indicator_dim: int = 10,
        lstm_hidden: int = 768,
        mlp_hidden: int = mlp_layer,
        dropout: float = 0.1,
        use_distance: bool = True,
        pos_dim: int = 50,
        max_distance: int = 128,
        num_roles: int = 6,
        role_dim: int = 32,
        attn_dim: int = 256,
    ):
        super().__init__()
        self.config = AutoConfig.from_pretrained(bert_name)
        self.bert = AutoModel.from_pretrained(bert_name)

        self.use_indicator = use_indicator
        self.use_distance = use_distance
        self.max_distance = max_distance

        bert_dim = self.config.hidden_size
        in_dim = bert_dim + (indicator_dim if use_indicator else 0)

        # =========================================================
        # # Role indicator embeddings
        # =========================================================
        if use_indicator:
            self.indicator_emb = nn.Embedding(num_roles, indicator_dim)

        # =========================================================
        # # Relative distance embeddings from predicate
        # =========================================================
        if use_distance:
            self.pos_emb = nn.Embedding(2 * max_distance + 1, pos_dim)
            in_dim += pos_dim

        # =========================================================
        # # BiLSTM over word-level sequence
        # =========================================================
        self.bilstm = nn.LSTM(
            input_size=in_dim,
            hidden_size=lstm_hidden // 2,
            num_layers=1,
            bidirectional=True,
            batch_first=True,
        )

        self.dropout = nn.Dropout(dropout)

        # =========================================================
        # # Directional attention
        # =========================================================
        self.W_q = nn.Linear(lstm_hidden, attn_dim)
        self.W_k = nn.Linear(lstm_hidden, attn_dim)
        self.W_v = nn.Linear(lstm_hidden, attn_dim)
        self.attn_layer_norm = nn.LayerNorm(attn_dim)

        # =========================================================
        # # Global utterance projection
        # =========================================================
        self.global_proj = nn.Linear(bert_dim, attn_dim)

        # =========================================================
        # # NEW: fusion modules
        # =========================================================
        # # Structural SRL representation from [query_source ; context]
        self.struct_proj = nn.Sequential(
            nn.Linear(lstm_hidden + attn_dim, lstm_hidden),
            nn.LayerNorm(lstm_hidden),
            nn.Tanh()
        )

        # # Project global utterance representation into same hidden space
        self.global_to_hidden = nn.Linear(attn_dim, lstm_hidden)

        # # Gate conditioned on [query_source ; context ; global_f]
        self.fusion_gate = nn.Linear(lstm_hidden + attn_dim + attn_dim, lstm_hidden)

        # =========================================================
        # # Frame scorer from fused representation
        # =========================================================
        self.frame_gate = nn.Sequential(
            nn.Linear(lstm_hidden, mlp_hidden),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 1)
        )

        # =========================================================
        # # Final formality head from fused representation
        # =========================================================
        self.formality_head = nn.Sequential(
            nn.Linear(lstm_hidden, mlp_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 1),
        )

        # =========================================================
        # # Fallback head when no frame information exists
        # =========================================================
        self.global_only_head = nn.Sequential(
            nn.Linear(bert_dim, mlp_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 1),
        )

    def masked_mean(self, x, mask):
        # x: [B, N, D], mask: [B, N]
        mask = mask.float()
        denom = mask.sum(dim=1, keepdim=True).clamp(min=1.0)
        return (x * mask.unsqueeze(-1)).sum(dim=1) / denom

    def forward(
        self,
        input_ids,
        token_type_ids,
        attention_mask,
        word_first_wp_fullidx,
        sentence_mask,
        sent_lens,
        frames_pred_word_idx=None,
        frames_role_ids=None,
        frames_arg0_mask=None,
        frames_arg1_mask=None,
        frames_argm_mask=None,   # kept for interface compatibility, not used in target attention
        frames_mask=None,
        formality=None,
        **kwargs
    ):
        device = input_ids.device
        B, _ = input_ids.size()
        N = sentence_mask.size(1)

        # =========================================================
        # # 1) BERT once per utterance
        # =========================================================
        bert_out = self.bert(
            input_ids=input_ids,
            token_type_ids=token_type_ids,
            attention_mask=attention_mask,
        )
        H = bert_out.last_hidden_state  # [B, T, D]

        # =========================================================
        # # Map first wordpiece positions back to word-level representations
        # =========================================================
        gather_idx = word_first_wp_fullidx.clone()
        gather_idx[gather_idx < 0] = 0
        gather_idx = gather_idx.unsqueeze(-1).expand(-1, -1, H.size(-1))
        H_words = torch.gather(H, dim=1, index=gather_idx)
        H_words = H_words * sentence_mask.unsqueeze(-1)

        # =========================================================
        # # Global utterance vector
        # =========================================================
        global_vec = self.masked_mean(H_words, sentence_mask)  # [B, D]

        # =========================================================
        # # Fallback path when there is no frame information
        # =========================================================
        if frames_mask is None or frames_pred_word_idx is None:
            score = self.global_only_head(self.dropout(global_vec)).squeeze(-1)

            loss = torch.tensor(0.0, device=device)
            if formality is not None:
                loss = nn.BCEWithLogitsLoss()(score, formality)
            return score, loss, torch.zeros((B, 1, N), device=device)

        # =========================================================
        # # 2) Flatten frames
        # =========================================================
        F = frames_mask.size(1)
        BF = B * F
        D = H_words.size(-1)

        Hf = H_words.unsqueeze(1).expand(B, F, N, D).contiguous().view(BF, N, D)
        sent_mask_f = sentence_mask.unsqueeze(1).expand(B, F, N).contiguous().view(BF, N)
        lengths_f = sent_lens.unsqueeze(1).expand(B, F).contiguous().view(BF)
        pred_idx_f = frames_pred_word_idx.contiguous().view(BF)

        X = Hf

        # =========================================================
        # # Add role indicator embeddings
        # =========================================================
        if self.use_indicator and frames_role_ids is not None:
            role_ids_f = frames_role_ids.clamp(0, 5).contiguous().view(BF, N)
            X = torch.cat([X, self.indicator_emb(role_ids_f)], dim=-1)

        # =========================================================
        # # Add relative distance embeddings from predicate
        # =========================================================
        if self.use_distance:
            positions = torch.arange(N, device=device).unsqueeze(0).expand(BF, -1)
            rel = (positions - pred_idx_f.unsqueeze(1)).clamp(-self.max_distance, self.max_distance)
            rel = rel + self.max_distance
            X = torch.cat([X, self.pos_emb(rel)], dim=-1)

        # =========================================================
        # # 3) BiLSTM over valid frames only
        # =========================================================
        valid_frame_mask = frames_mask.reshape(-1).bool()

        if valid_frame_mask.sum() == 0:
            score = self.global_only_head(self.dropout(global_vec)).squeeze(-1)

            loss = torch.tensor(0.0, device=device)
            if formality is not None:
                loss = nn.BCEWithLogitsLoss()(score, formality)
            return score, loss, torch.zeros((B, 1, N), device=device)

        X_valid = X[valid_frame_mask]
        lengths_valid = lengths_f[valid_frame_mask].clamp(min=1)

        packed = pack_padded_sequence(
            X_valid,
            lengths=lengths_valid.cpu(),
            batch_first=True,
            enforce_sorted=False
        )
        G_packed, _ = self.bilstm(packed)
        G_valid, _ = pad_packed_sequence(G_packed, batch_first=True, total_length=N)
        G_valid = self.dropout(G_valid)

        H_lstm = G_valid.size(-1)
        G = torch.zeros((BF, N, H_lstm), device=device, dtype=G_valid.dtype)
        G[valid_frame_mask] = G_valid

        # =========================================================
        # # 4) Directional attention
        # =========================================================
        arg0_f = frames_arg0_mask.contiguous().view(BF, N) if frames_arg0_mask is not None else None
        arg1_f = frames_arg1_mask.contiguous().view(BF, N) if frames_arg1_mask is not None else None

        batch_idx = torch.arange(BF, device=device)
        gp = G[batch_idx, pred_idx_f.clamp(min=0, max=N - 1), :]  # predicate vector

        # =========================================================
        # # Query from ARG0 mean, fallback to predicate
        # =========================================================
        if arg0_f is not None:
            arg0_sum = arg0_f.sum(dim=1, keepdim=True)
            has_arg0 = (arg0_sum > 0).float()
            arg0_vec = (G * arg0_f.unsqueeze(-1)).sum(dim=1) / arg0_sum.clamp(min=1.0)
            query_source = has_arg0 * arg0_vec + (1.0 - has_arg0) * gp
        else:
            query_source = gp

        # =========================================================
        # # CHANGED: attend to ARG1 only, otherwise full sentence
        # =========================================================
        if arg1_f is not None:
            arg1_f = arg1_f.float()
            has_arg1 = (arg1_f.sum(dim=1, keepdim=True) > 0).float()

            # # Use ARG1 when present; otherwise fall back to full sentence
            target_mask = has_arg1 * arg1_f + (1.0 - has_arg1) * sent_mask_f.float()
        else:
            # # If ARG1 is unavailable entirely, fall back to full sentence
            target_mask = sent_mask_f.float()

        target_mask = target_mask * sent_mask_f.float()

        # =========================================================
        # # Directional attention computation
        # =========================================================
        Q = self.W_q(query_source).unsqueeze(1)   # [BF, 1, A]
        K = self.W_k(G)                           # [BF, N, A]
        V = self.W_v(G)                           # [BF, N, A]

        attn_scores = torch.bmm(Q, K.transpose(1, 2)) / (K.size(-1) ** 0.5)
        attn_scores = attn_scores.masked_fill(target_mask.unsqueeze(1) == 0, -10000.0)
        attn_weights = torch.softmax(attn_scores, dim=-1)
        context = torch.bmm(attn_weights, V).squeeze(1)
        context = self.attn_layer_norm(context)

        # =========================================================
        # # Global vector expanded to frames
        # =========================================================
        global_f = global_vec.unsqueeze(1).expand(B, F, D).contiguous().view(BF, D)
        global_f = self.global_proj(global_f)   # [BF, attn_dim]

        # =========================================================
        # # 5) Gated fusion between global text and directional SRL
        # =========================================================
        # # Structural signal = [ARG0/predicate query ; ARG1-directed context]
        struct_input = torch.cat([query_source, context], dim=1)      # [BF, lstm_hidden + attn_dim]
        struct_h = self.struct_proj(self.dropout(struct_input))       # [BF, lstm_hidden]

        # # Global text signal projected to same hidden space
        global_h = self.global_to_hidden(self.dropout(global_f))      # [BF, lstm_hidden]

        # # Gate conditioned on global + directional features
        gate_input = torch.cat([query_source, context, global_f], dim=1)
        fusion_gate = torch.sigmoid(self.fusion_gate(self.dropout(gate_input)))  # [BF, lstm_hidden]

        # # Residual fusion: global text updated by directional SRL
        fused_frame = global_h + fusion_gate * struct_h               # [BF, lstm_hidden]

        # =========================================================
        # # 6) Frame score and frame importance from fused representation
        # =========================================================
        score_frame = self.formality_head(self.dropout(fused_frame)).squeeze(-1)
        gate_frame = self.frame_gate(self.dropout(fused_frame)).squeeze(-1)

        # =========================================================
        # # 7) Aggregate frames with learned weights
        # =========================================================
        score_frame = score_frame.view(B, F)
        gate_frame = gate_frame.view(B, F)

        fm = frames_mask.float()
        gate_frame = gate_frame.masked_fill(fm == 0, -10000.0)
        frame_alpha = torch.softmax(gate_frame, dim=1) * fm
        frame_alpha = frame_alpha / frame_alpha.sum(dim=1, keepdim=True).clamp(min=1e-8)

        score_utt = (score_frame * frame_alpha).sum(dim=1)

        attn_weights = attn_weights.squeeze(1).view(B, F, N)
        attn_weights = attn_weights * fm.unsqueeze(-1)

        loss = torch.tensor(0.0, device=device)
        if formality is not None:
            loss = nn.BCEWithLogitsLoss()(score_utt, formality)

        return score_utt, loss, attn_weights

### ARG-> ARG1 fusion no MLP

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoConfig, AutoModel
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

# Fusion no mlp 
class DirectionalSRL_fusionnomlp(nn.Module):
    """
    SRL-aware BERT model for formality prediction with directional SRL + gated fusion.

    Directional design:
      - Query primarily from ARG0, fallback to predicate
      - Attend to ARG1 only if present, otherwise full sentence
      - Add global utterance representation
      - Learn frame importance instead of uniform averaging

    Fusion design:
      - Build a directional SRL representation from [query_source ; context]
      - Build a global text representation from the utterance
      - Use gated residual fusion so SRL selectively updates the global text representation
    """

    def __init__(
        self,
        bert_name: str,
        num_labels: int,
        use_indicator: bool = True,
        indicator_dim: int = 10,
        lstm_hidden: int = 768,
        mlp_hidden: int = mlp_layer,  # kept for compatibility; not used now
        dropout: float = 0.1,
        use_distance: bool = True,
        pos_dim: int = 50,
        max_distance: int = 128,
        num_roles: int = 6,
        role_dim: int = 32,
        attn_dim: int = 256,
    ):
        super().__init__()
        self.config = AutoConfig.from_pretrained(bert_name)
        self.bert = AutoModel.from_pretrained(bert_name)

        self.use_indicator = use_indicator
        self.use_distance = use_distance
        self.max_distance = max_distance

        bert_dim = self.config.hidden_size
        in_dim = bert_dim + (indicator_dim if use_indicator else 0)

        # =========================================================
        # # Role indicator embeddings
        # =========================================================
        if use_indicator:
            self.indicator_emb = nn.Embedding(num_roles, indicator_dim)

        # =========================================================
        # # Relative distance embeddings from predicate
        # =========================================================
        if use_distance:
            self.pos_emb = nn.Embedding(2 * max_distance + 1, pos_dim)
            in_dim += pos_dim

        # =========================================================
        # # BiLSTM over word-level sequence
        # =========================================================
        self.bilstm = nn.LSTM(
            input_size=in_dim,
            hidden_size=lstm_hidden // 2,
            num_layers=1,
            bidirectional=True,
            batch_first=True,
        )

        self.dropout = nn.Dropout(dropout)

        # =========================================================
        # # Directional attention
        # =========================================================
        self.W_q = nn.Linear(lstm_hidden, attn_dim)
        self.W_k = nn.Linear(lstm_hidden, attn_dim)
        self.W_v = nn.Linear(lstm_hidden, attn_dim)
        self.attn_layer_norm = nn.LayerNorm(attn_dim)

        # =========================================================
        # # Global utterance projection
        # =========================================================
        self.global_proj = nn.Linear(bert_dim, attn_dim)

        # =========================================================
        # # Fusion modules
        # =========================================================
        self.struct_proj = nn.Sequential(
            nn.Linear(lstm_hidden + attn_dim, lstm_hidden),
            nn.LayerNorm(lstm_hidden),
            nn.Tanh()
        )

        self.global_to_hidden = nn.Linear(attn_dim, lstm_hidden)
        self.fusion_gate = nn.Linear(lstm_hidden + attn_dim + attn_dim, lstm_hidden)

        # =========================================================
        # # CHANGED: remove extra MLP from frame scorer
        # # before: Linear -> Tanh -> Dropout -> Linear
        # # now: simple linear scorer
        # =========================================================
        self.frame_gate = nn.Linear(lstm_hidden, 1)

        # =========================================================
        # # CHANGED: remove extra MLP from final formality head
        # # before: Linear -> ReLU -> Dropout -> Linear
        # # now: simple linear head
        # =========================================================
        self.formality_head = nn.Linear(lstm_hidden, 1)

        # =========================================================
        # # CHANGED: remove extra MLP from fallback head
        # =========================================================
        self.global_only_head = nn.Linear(bert_dim, 1)

    def masked_mean(self, x, mask):
        # x: [B, N, D], mask: [B, N]
        mask = mask.float()
        denom = mask.sum(dim=1, keepdim=True).clamp(min=1.0)
        return (x * mask.unsqueeze(-1)).sum(dim=1) / denom

    def forward(
        self,
        input_ids,
        token_type_ids,
        attention_mask,
        word_first_wp_fullidx,
        sentence_mask,
        sent_lens,
        frames_pred_word_idx=None,
        frames_role_ids=None,
        frames_arg0_mask=None,
        frames_arg1_mask=None,
        frames_argm_mask=None,   # kept for interface compatibility, not used in target attention
        frames_mask=None,
        formality=None,
        **kwargs
    ):
        device = input_ids.device
        B, _ = input_ids.size()
        N = sentence_mask.size(1)

        # =========================================================
        # # 1) BERT once per utterance
        # =========================================================
        bert_out = self.bert(
            input_ids=input_ids,
            token_type_ids=token_type_ids,
            attention_mask=attention_mask,
        )
        H = bert_out.last_hidden_state  # [B, T, D]

        # =========================================================
        # # Map first wordpiece positions back to word-level representations
        # =========================================================
        gather_idx = word_first_wp_fullidx.clone()
        gather_idx[gather_idx < 0] = 0
        gather_idx = gather_idx.unsqueeze(-1).expand(-1, -1, H.size(-1))
        H_words = torch.gather(H, dim=1, index=gather_idx)
        H_words = H_words * sentence_mask.unsqueeze(-1)

        # =========================================================
        # # Global utterance vector
        # =========================================================
        global_vec = self.masked_mean(H_words, sentence_mask)  # [B, D]

        # =========================================================
        # # Fallback path when there is no frame information
        # =========================================================
        if frames_mask is None or frames_pred_word_idx is None:
            score = self.global_only_head(self.dropout(global_vec)).squeeze(-1)

            loss = torch.tensor(0.0, device=device)
            if formality is not None:
                loss = nn.BCEWithLogitsLoss()(score, formality)
            return score, loss, torch.zeros((B, 1, N), device=device)

        # =========================================================
        # # 2) Flatten frames
        # =========================================================
        F = frames_mask.size(1)
        BF = B * F
        D = H_words.size(-1)

        Hf = H_words.unsqueeze(1).expand(B, F, N, D).contiguous().view(BF, N, D)
        sent_mask_f = sentence_mask.unsqueeze(1).expand(B, F, N).contiguous().view(BF, N)
        lengths_f = sent_lens.unsqueeze(1).expand(B, F).contiguous().view(BF)
        pred_idx_f = frames_pred_word_idx.contiguous().view(BF)

        X = Hf

        # =========================================================
        # # Add role indicator embeddings
        # =========================================================
        if self.use_indicator and frames_role_ids is not None:
            role_ids_f = frames_role_ids.clamp(0, 5).contiguous().view(BF, N)
            X = torch.cat([X, self.indicator_emb(role_ids_f)], dim=-1)

        # =========================================================
        # # Add relative distance embeddings from predicate
        # =========================================================
        if self.use_distance:
            positions = torch.arange(N, device=device).unsqueeze(0).expand(BF, -1)
            rel = (positions - pred_idx_f.unsqueeze(1)).clamp(-self.max_distance, self.max_distance)
            rel = rel + self.max_distance
            X = torch.cat([X, self.pos_emb(rel)], dim=-1)

        # =========================================================
        # # 3) BiLSTM over valid frames only
        # =========================================================
        valid_frame_mask = frames_mask.reshape(-1).bool()

        if valid_frame_mask.sum() == 0:
            score = self.global_only_head(self.dropout(global_vec)).squeeze(-1)

            loss = torch.tensor(0.0, device=device)
            if formality is not None:
                loss = nn.BCEWithLogitsLoss()(score, formality)
            return score, loss, torch.zeros((B, 1, N), device=device)

        X_valid = X[valid_frame_mask]
        lengths_valid = lengths_f[valid_frame_mask].clamp(min=1)

        packed = pack_padded_sequence(
            X_valid,
            lengths=lengths_valid.cpu(),
            batch_first=True,
            enforce_sorted=False
        )
        G_packed, _ = self.bilstm(packed)
        G_valid, _ = pad_packed_sequence(G_packed, batch_first=True, total_length=N)
        G_valid = self.dropout(G_valid)

        H_lstm = G_valid.size(-1)
        G = torch.zeros((BF, N, H_lstm), device=device, dtype=G_valid.dtype)
        G[valid_frame_mask] = G_valid

        # =========================================================
        # # 4) Directional attention
        # =========================================================
        arg0_f = frames_arg0_mask.contiguous().view(BF, N) if frames_arg0_mask is not None else None
        arg1_f = frames_arg1_mask.contiguous().view(BF, N) if frames_arg1_mask is not None else None

        batch_idx = torch.arange(BF, device=device)
        gp = G[batch_idx, pred_idx_f.clamp(min=0, max=N - 1), :]  # predicate vector

        # =========================================================
        # # Query from ARG0 mean, fallback to predicate
        # =========================================================
        if arg0_f is not None:
            arg0_sum = arg0_f.sum(dim=1, keepdim=True)
            has_arg0 = (arg0_sum > 0).float()
            arg0_vec = (G * arg0_f.unsqueeze(-1)).sum(dim=1) / arg0_sum.clamp(min=1.0)
            query_source = has_arg0 * arg0_vec + (1.0 - has_arg0) * gp
        else:
            query_source = gp

        # =========================================================
        # # Attend to ARG1 only, otherwise full sentence
        # =========================================================
        if arg1_f is not None:
            arg1_f = arg1_f.float()
            has_arg1 = (arg1_f.sum(dim=1, keepdim=True) > 0).float()
            target_mask = has_arg1 * arg1_f + (1.0 - has_arg1) * sent_mask_f.float()
        else:
            target_mask = sent_mask_f.float()

        target_mask = target_mask * sent_mask_f.float()

        # =========================================================
        # # Directional attention computation
        # =========================================================
        Q = self.W_q(query_source).unsqueeze(1)   # [BF, 1, A]
        K = self.W_k(G)                           # [BF, N, A]
        V = self.W_v(G)                           # [BF, N, A]

        attn_scores = torch.bmm(Q, K.transpose(1, 2)) / (K.size(-1) ** 0.5)
        attn_scores = attn_scores.masked_fill(target_mask.unsqueeze(1) == 0, -10000.0)
        attn_weights = torch.softmax(attn_scores, dim=-1)
        context = torch.bmm(attn_weights, V).squeeze(1)
        context = self.attn_layer_norm(context)

        # =========================================================
        # # Global vector expanded to frames
        # =========================================================
        global_f = global_vec.unsqueeze(1).expand(B, F, D).contiguous().view(BF, D)
        global_f = self.global_proj(global_f)   # [BF, attn_dim]

        # =========================================================
        # # 5) Gated fusion between global text and directional SRL
        # =========================================================
        struct_input = torch.cat([query_source, context], dim=1)      # [BF, lstm_hidden + attn_dim]
        struct_h = self.struct_proj(self.dropout(struct_input))       # [BF, lstm_hidden]

        global_h = self.global_to_hidden(self.dropout(global_f))      # [BF, lstm_hidden]

        gate_input = torch.cat([query_source, context, global_f], dim=1)
        fusion_gate = torch.sigmoid(self.fusion_gate(self.dropout(gate_input)))  # [BF, lstm_hidden]

        fused_frame = global_h + fusion_gate * struct_h               # [BF, lstm_hidden]

        # =========================================================
        # # 6) Frame score and frame importance from fused representation
        # =========================================================
        score_frame = self.formality_head(self.dropout(fused_frame)).squeeze(-1)
        gate_frame = self.frame_gate(self.dropout(fused_frame)).squeeze(-1)

        # =========================================================
        # # 7) Aggregate frames with learned weights
        # =========================================================
        score_frame = score_frame.view(B, F)
        gate_frame = gate_frame.view(B, F)

        fm = frames_mask.float()
        gate_frame = gate_frame.masked_fill(fm == 0, -10000.0)
        frame_alpha = torch.softmax(gate_frame, dim=1) * fm
        frame_alpha = frame_alpha / frame_alpha.sum(dim=1, keepdim=True).clamp(min=1e-8)

        score_utt = (score_frame * frame_alpha).sum(dim=1)

        attn_weights = attn_weights.squeeze(1).view(B, F, N)
        attn_weights = attn_weights * fm.unsqueeze(-1)

        loss = torch.tensor(0.0, device=device)
        if formality is not None:
            loss = nn.BCEWithLogitsLoss()(score_utt, formality)

        return score_utt, loss, attn_weights

### ARG0 -> ARG1 Last concat mlp

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoConfig, AutoModel
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence


class DirectionalSRL_lastlayermlp(nn.Module):
    """
    SRL-aware BERT model for formality prediction.

    Directional design:
      - Query primarily from ARG0, fallback to predicate
      - Attend to ARG1 only if present, otherwise full sentence
      - Add global utterance representation
      - Learn frame importance instead of uniform averaging

    Last-layer concatenation design:
      - Use the original directional pipeline
      - Replace fusion with direct concatenation at the last layer
    """

    def __init__(
        self,
        bert_name: str,
        num_labels: int,
        use_indicator: bool = True,
        indicator_dim: int = 10,
        lstm_hidden: int = 768,
        mlp_hidden: int = 300,
        dropout: float = 0.1,
        use_distance: bool = True,
        pos_dim: int = 50,
        max_distance: int = 128,
        num_roles: int = 6,
        role_dim: int = 32,
        attn_dim: int = 256,
    ):
        super().__init__()
        self.config = AutoConfig.from_pretrained(bert_name)
        self.bert = AutoModel.from_pretrained(bert_name)

        self.use_indicator = use_indicator
        self.use_distance = use_distance
        self.max_distance = max_distance

        bert_dim = self.config.hidden_size
        in_dim = bert_dim + (indicator_dim if use_indicator else 0)

        if use_indicator:
            self.indicator_emb = nn.Embedding(num_roles, indicator_dim)

        if use_distance:
            self.pos_emb = nn.Embedding(2 * max_distance + 1, pos_dim)
            in_dim += pos_dim

        self.bilstm = nn.LSTM(
            input_size=in_dim,
            hidden_size=lstm_hidden // 2,
            num_layers=1,
            bidirectional=True,
            batch_first=True,
        )

        self.dropout = nn.Dropout(dropout)

        # directional attention
        self.W_q = nn.Linear(lstm_hidden, attn_dim)
        self.W_k = nn.Linear(lstm_hidden, attn_dim)
        self.W_v = nn.Linear(lstm_hidden, attn_dim)
        self.attn_layer_norm = nn.LayerNorm(attn_dim)

        # keep original global projection style
        self.global_proj = nn.Linear(lstm_hidden, attn_dim)

        concat_dim = lstm_hidden + attn_dim + attn_dim

        # frame scorer for learned frame aggregation
        self.frame_gate = nn.Sequential(
            nn.Linear(concat_dim, mlp_hidden),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 1)
        )

        # final head
        self.formality_head = nn.Sequential(
            nn.Linear(concat_dim, mlp_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 1),
        )

    def masked_mean(self, x, mask):
        mask = mask.float()
        denom = mask.sum(dim=1, keepdim=True).clamp(min=1.0)
        return (x * mask.unsqueeze(-1)).sum(dim=1) / denom

    def forward(
        self,
        input_ids,
        token_type_ids,
        attention_mask,
        word_first_wp_fullidx,
        sentence_mask,
        sent_lens,
        frames_pred_word_idx=None,
        frames_role_ids=None,
        frames_arg0_mask=None,
        frames_arg1_mask=None,
        frames_argm_mask=None,   # kept for compatibility, not used
        frames_mask=None,
        formality=None,
        **kwargs
    ):
        device = input_ids.device
        B, _ = input_ids.size()
        N = sentence_mask.size(1)

        # 1) BERT once per utterance
        bert_out = self.bert(
            input_ids=input_ids,
            token_type_ids=token_type_ids,
            attention_mask=attention_mask,
        )
        H = bert_out.last_hidden_state  # [B, T, D]

        # map first wordpiece positions back to word-level representations
        gather_idx = word_first_wp_fullidx.clone()
        gather_idx[gather_idx < 0] = 0
        gather_idx = gather_idx.unsqueeze(-1).expand(-1, -1, H.size(-1))
        H_words = torch.gather(H, dim=1, index=gather_idx)
        H_words = H_words * sentence_mask.unsqueeze(-1)

        # global utterance vector
        global_vec = self.masked_mean(H_words, sentence_mask)  # [B, D]

        # fallback path if no frame info
        if frames_mask is None or frames_pred_word_idx is None:
            pooled = self.masked_mean(H_words, sentence_mask)
            global_proj = self.global_proj(pooled)
            score = self.formality_head(
                torch.cat([pooled, global_proj, global_proj], dim=-1)
            ).squeeze(-1)

            loss = torch.tensor(0.0, device=device)
            if formality is not None:
                loss = nn.BCEWithLogitsLoss()(score, formality)
            return score, loss, torch.zeros((B, 1, N), device=device)

        # 2) flatten frames
        F = frames_mask.size(1)
        BF = B * F
        D = H_words.size(-1)

        Hf = H_words.unsqueeze(1).expand(B, F, N, D).contiguous().view(BF, N, D)
        sent_mask_f = sentence_mask.unsqueeze(1).expand(B, F, N).contiguous().view(BF, N)
        lengths_f = sent_lens.unsqueeze(1).expand(B, F).contiguous().view(BF)
        pred_idx_f = frames_pred_word_idx.contiguous().view(BF)

        X = Hf

        if self.use_indicator and frames_role_ids is not None:
            role_ids_f = frames_role_ids.clamp(0, 5).contiguous().view(BF, N)
            X = torch.cat([X, self.indicator_emb(role_ids_f)], dim=-1)

        if self.use_distance:
            positions = torch.arange(N, device=device).unsqueeze(0).expand(BF, -1)
            rel = (positions - pred_idx_f.unsqueeze(1)).clamp(-self.max_distance, self.max_distance)
            rel = rel + self.max_distance
            X = torch.cat([X, self.pos_emb(rel)], dim=-1)

        # 3) BiLSTM over valid frames only
        valid_frame_mask = frames_mask.reshape(-1).bool()

        if valid_frame_mask.sum() == 0:
            pooled = self.masked_mean(H_words, sentence_mask)
            global_proj = self.global_proj(pooled)
            score = self.formality_head(
                torch.cat([pooled, global_proj, global_proj], dim=-1)
            ).squeeze(-1)

            loss = torch.tensor(0.0, device=device)
            if formality is not None:
                loss = nn.BCEWithLogitsLoss()(score, formality)
            return score, loss, torch.zeros((B, 1, N), device=device)

        X_valid = X[valid_frame_mask]
        lengths_valid = lengths_f[valid_frame_mask].clamp(min=1)

        packed = pack_padded_sequence(
            X_valid,
            lengths=lengths_valid.cpu(),
            batch_first=True,
            enforce_sorted=False
        )
        G_packed, _ = self.bilstm(packed)
        G_valid, _ = pad_packed_sequence(G_packed, batch_first=True, total_length=N)
        G_valid = self.dropout(G_valid)

        H_lstm = G_valid.size(-1)
        G = torch.zeros((BF, N, H_lstm), device=device, dtype=G_valid.dtype)
        G[valid_frame_mask] = G_valid

        # 4) directional attention: ARG0 -> ARG1 only
        arg0_f = frames_arg0_mask.contiguous().view(BF, N) if frames_arg0_mask is not None else None
        arg1_f = frames_arg1_mask.contiguous().view(BF, N) if frames_arg1_mask is not None else None

        batch_idx = torch.arange(BF, device=device)
        gp = G[batch_idx, pred_idx_f.clamp(min=0, max=N - 1), :]  # predicate vector

        # query from ARG0 mean, fallback to predicate
        if arg0_f is not None:
            arg0_sum = arg0_f.sum(dim=1, keepdim=True)
            has_arg0 = (arg0_sum > 0).float()
            arg0_vec = (G * arg0_f.unsqueeze(-1)).sum(dim=1) / arg0_sum.clamp(min=1.0)
            query_source = has_arg0 * arg0_vec + (1.0 - has_arg0) * gp
        else:
            query_source = gp

        # attend to ARG1 only, otherwise full sentence
        if arg1_f is not None:
            arg1_f = arg1_f.float()
            has_arg1 = (arg1_f.sum(dim=1, keepdim=True) > 0).float()
            target_mask = has_arg1 * arg1_f + (1.0 - has_arg1) * sent_mask_f.float()
        else:
            target_mask = sent_mask_f.float()

        target_mask = target_mask * sent_mask_f.float()

        Q = self.W_q(query_source).unsqueeze(1)   # [BF, 1, A]
        K = self.W_k(G)                           # [BF, N, A]
        V = self.W_v(G)                           # [BF, N, A]

        attn_scores = torch.bmm(Q, K.transpose(1, 2)) / (K.size(-1) ** 0.5)
        attn_scores = attn_scores.masked_fill(target_mask.unsqueeze(1) == 0, -10000.0)
        attn_weights = torch.softmax(attn_scores, dim=-1)
        context = torch.bmm(attn_weights, V).squeeze(1)
        context = self.attn_layer_norm(context)

        # global vector expanded to frames
        global_f = global_vec.unsqueeze(1).expand(B, F, D).contiguous().view(BF, D)
        global_f = self.global_proj(global_f)

        # only change from fusion: last-layer concatenation
        frame_features = torch.cat([query_source, context, global_f], dim=1)

        score_frame = self.formality_head(frame_features).squeeze(-1)
        gate_frame = self.frame_gate(frame_features).squeeze(-1)

        # 5) aggregate frames with learned weights
        score_frame = score_frame.view(B, F)
        gate_frame = gate_frame.view(B, F)

        fm = frames_mask.float()
        gate_frame = gate_frame.masked_fill(fm == 0, -10000.0)
        frame_alpha = torch.softmax(gate_frame, dim=1) * fm
        frame_alpha = frame_alpha / frame_alpha.sum(dim=1, keepdim=True).clamp(min=1e-8)

        score_utt = (score_frame * frame_alpha).sum(dim=1)

        attn_weights = attn_weights.squeeze(1).view(B, F, N)
        attn_weights = attn_weights * fm.unsqueeze(-1)

        loss = torch.tensor(0.0, device=device)
        if formality is not None:
            loss = nn.BCEWithLogitsLoss()(score_utt, formality)

        return score_utt, loss, attn_weights

### ARG0 -> ARG1 Last concat no mlp

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoConfig, AutoModel
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence


class DirectionalSRL_lastlayernomlp(nn.Module):
    """
    SRL-aware BERT model for formality prediction.
    Same as the original directional version, except the final integration uses
    last-layer concatenation and the heads do not use an extra MLP.
    """

    def __init__(
        self,
        bert_name: str,
        num_labels: int,
        use_indicator: bool = True,
        indicator_dim: int = 10,
        lstm_hidden: int = 768,
        mlp_hidden: int = 300,
        dropout: float = 0.1,
        use_distance: bool = True,
        pos_dim: int = 50,
        max_distance: int = 128,
        num_roles: int = 6,
        role_dim: int = 32,
        attn_dim: int = 256,
    ):
        super().__init__()
        self.config = AutoConfig.from_pretrained(bert_name)
        self.bert = AutoModel.from_pretrained(bert_name)

        self.use_indicator = use_indicator
        self.use_distance = use_distance
        self.max_distance = max_distance

        bert_dim = self.config.hidden_size
        in_dim = bert_dim + (indicator_dim if use_indicator else 0)

        if use_indicator:
            self.indicator_emb = nn.Embedding(num_roles, indicator_dim)

        if use_distance:
            self.pos_emb = nn.Embedding(2 * max_distance + 1, pos_dim)
            in_dim += pos_dim

        self.bilstm = nn.LSTM(
            input_size=in_dim,
            hidden_size=lstm_hidden // 2,
            num_layers=1,
            bidirectional=True,
            batch_first=True,
        )

        self.dropout = nn.Dropout(dropout)

        # directional attention
        self.W_q = nn.Linear(lstm_hidden, attn_dim)
        self.W_k = nn.Linear(lstm_hidden, attn_dim)
        self.W_v = nn.Linear(lstm_hidden, attn_dim)
        self.attn_layer_norm = nn.LayerNorm(attn_dim)

        # global utterance pooling
        self.global_proj = nn.Linear(lstm_hidden, attn_dim)

        concat_dim = lstm_hidden + attn_dim + attn_dim

        self.frame_gate = nn.Linear(concat_dim, 1)
        self.formality_head = nn.Linear(concat_dim, 1)

    def masked_mean(self, x, mask):
        mask = mask.float()
        denom = mask.sum(dim=1, keepdim=True).clamp(min=1.0)
        return (x * mask.unsqueeze(-1)).sum(dim=1) / denom

    def forward(
        self,
        input_ids,
        token_type_ids,
        attention_mask,
        word_first_wp_fullidx,
        sentence_mask,
        sent_lens,
        frames_pred_word_idx=None,
        frames_role_ids=None,
        frames_arg0_mask=None,
        frames_arg1_mask=None,
        frames_argm_mask=None,   # kept for compatibility, not used
        frames_mask=None,
        formality=None,
        **kwargs
    ):
        device = input_ids.device
        B, _ = input_ids.size()
        N = sentence_mask.size(1)

        bert_out = self.bert(
            input_ids=input_ids,
            token_type_ids=token_type_ids,
            attention_mask=attention_mask,
        )
        H = bert_out.last_hidden_state

        gather_idx = word_first_wp_fullidx.clone()
        gather_idx[gather_idx < 0] = 0
        gather_idx = gather_idx.unsqueeze(-1).expand(-1, -1, H.size(-1))
        H_words = torch.gather(H, dim=1, index=gather_idx)
        H_words = H_words * sentence_mask.unsqueeze(-1)

        global_vec = self.masked_mean(H_words, sentence_mask)

        if frames_mask is None or frames_pred_word_idx is None:
            pooled = self.masked_mean(H_words, sentence_mask)
            global_proj = self.global_proj(pooled)
            score = self.formality_head(
                torch.cat([pooled, global_proj, global_proj], dim=-1)
            ).squeeze(-1)

            loss = torch.tensor(0.0, device=device)
            if formality is not None:
                loss = nn.BCEWithLogitsLoss()(score, formality)
            return score, loss, torch.zeros((B, 1, N), device=device)

        F = frames_mask.size(1)
        BF = B * F
        D = H_words.size(-1)

        Hf = H_words.unsqueeze(1).expand(B, F, N, D).contiguous().view(BF, N, D)
        sent_mask_f = sentence_mask.unsqueeze(1).expand(B, F, N).contiguous().view(BF, N)
        lengths_f = sent_lens.unsqueeze(1).expand(B, F).contiguous().view(BF)
        pred_idx_f = frames_pred_word_idx.contiguous().view(BF)

        X = Hf

        if self.use_indicator and frames_role_ids is not None:
            role_ids_f = frames_role_ids.clamp(0, 5).contiguous().view(BF, N)
            X = torch.cat([X, self.indicator_emb(role_ids_f)], dim=-1)

        if self.use_distance:
            positions = torch.arange(N, device=device).unsqueeze(0).expand(BF, -1)
            rel = (positions - pred_idx_f.unsqueeze(1)).clamp(-self.max_distance, self.max_distance)
            rel = rel + self.max_distance
            X = torch.cat([X, self.pos_emb(rel)], dim=-1)

        valid_frame_mask = frames_mask.reshape(-1).bool()

        if valid_frame_mask.sum() == 0:
            pooled = self.masked_mean(H_words, sentence_mask)
            global_proj = self.global_proj(pooled)
            score = self.formality_head(
                torch.cat([pooled, global_proj, global_proj], dim=-1)
            ).squeeze(-1)

            loss = torch.tensor(0.0, device=device)
            if formality is not None:
                loss = nn.BCEWithLogitsLoss()(score, formality)
            return score, loss, torch.zeros((B, 1, N), device=device)

        X_valid = X[valid_frame_mask]
        lengths_valid = lengths_f[valid_frame_mask].clamp(min=1)

        packed = pack_padded_sequence(
            X_valid,
            lengths=lengths_valid.cpu(),
            batch_first=True,
            enforce_sorted=False
        )
        G_packed, _ = self.bilstm(packed)
        G_valid, _ = pad_packed_sequence(G_packed, batch_first=True, total_length=N)
        G_valid = self.dropout(G_valid)

        H_lstm = G_valid.size(-1)
        G = torch.zeros((BF, N, H_lstm), device=device, dtype=G_valid.dtype)
        G[valid_frame_mask] = G_valid

        # directional attention: ARG0 -> ARG1 only
        arg0_f = frames_arg0_mask.contiguous().view(BF, N) if frames_arg0_mask is not None else None
        arg1_f = frames_arg1_mask.contiguous().view(BF, N) if frames_arg1_mask is not None else None

        batch_idx = torch.arange(BF, device=device)
        gp = G[batch_idx, pred_idx_f.clamp(min=0, max=N - 1), :]

        if arg0_f is not None:
            arg0_sum = arg0_f.sum(dim=1, keepdim=True)
            has_arg0 = (arg0_sum > 0).float()
            arg0_vec = (G * arg0_f.unsqueeze(-1)).sum(dim=1) / arg0_sum.clamp(min=1.0)
            query_source = has_arg0 * arg0_vec + (1.0 - has_arg0) * gp
        else:
            query_source = gp

        if arg1_f is not None:
            arg1_f = arg1_f.float()
            has_arg1 = (arg1_f.sum(dim=1, keepdim=True) > 0).float()
            target_mask = has_arg1 * arg1_f + (1.0 - has_arg1) * sent_mask_f.float()
        else:
            target_mask = sent_mask_f.float()

        target_mask = target_mask * sent_mask_f.float()

        Q = self.W_q(query_source).unsqueeze(1)
        K = self.W_k(G)
        V = self.W_v(G)

        attn_scores = torch.bmm(Q, K.transpose(1, 2)) / (K.size(-1) ** 0.5)
        attn_scores = attn_scores.masked_fill(target_mask.unsqueeze(1) == 0, -10000.0)
        attn_weights = torch.softmax(attn_scores, dim=-1)
        context = torch.bmm(attn_weights, V).squeeze(1)
        context = self.attn_layer_norm(context)

        global_f = global_vec.unsqueeze(1).expand(B, F, D).contiguous().view(BF, D)
        global_f = self.global_proj(global_f)

        frame_features = torch.cat([query_source, context, global_f], dim=1)

        score_frame = self.formality_head(frame_features).squeeze(-1)
        gate_frame = self.frame_gate(frame_features).squeeze(-1)

        score_frame = score_frame.view(B, F)
        gate_frame = gate_frame.view(B, F)

        fm = frames_mask.float()
        gate_frame = gate_frame.masked_fill(fm == 0, -10000.0)
        frame_alpha = torch.softmax(gate_frame, dim=1) * fm
        frame_alpha = frame_alpha / frame_alpha.sum(dim=1, keepdim=True).clamp(min=1e-8)

        score_utt = (score_frame * frame_alpha).sum(dim=1)

        attn_weights = attn_weights.squeeze(1).view(B, F, N)
        attn_weights = attn_weights * fm.unsqueeze(-1)

        loss = torch.tensor(0.0, device=device)
        if formality is not None:
            loss = nn.BCEWithLogitsLoss()(score_utt, formality)

        return score_utt, loss, attn_weights

# 4. Training Loops

In [ ]:
from scipy.stats import pearsonr

@torch.no_grad()
def eval_formality(model, dataloader, device="cuda"):
    model.eval()
    total_loss, n_batches = 0.0, 0
    all_preds, all_golds  = [], []

    for batch in dataloader:
        batch = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}
        preds, loss, _ = model(**batch)
        total_loss += float(loss.item())
        n_batches  += 1

        if "formality" in batch:
            all_preds.append(preds.detach().cpu())
            all_golds.append(batch["formality"].detach().cpu())

    avg_loss = total_loss / max(1, n_batches)
    if len(all_preds) > 0:
        all_preds = torch.cat(all_preds).numpy()
        all_golds = torch.cat(all_golds).numpy()
        probs     = torch.sigmoid(torch.tensor(all_preds)).numpy()
        binary    = (probs >= 0.5).astype(float)
        acc       = (binary == all_golds).mean()
    else:
        acc = 0.0
    return avg_loss, acc


def train_one_epoch(
    model,
    dataloader,
    optimizer,
    device="cuda",
    scheduler=None,
    grad_accum_steps=1,
    amp=True,
    max_grad_norm=1.0,
):
    model.train()
    total_loss, n_steps = 0.0, 0
    use_amp = amp and torch.cuda.is_available()
    scaler  = torch.amp.GradScaler('cuda', enabled=use_amp)
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(dataloader, 1):
        batch = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}
        with torch.amp.autocast('cuda', enabled=use_amp, dtype=torch.float16):
            _, loss, _ = model(**batch)
        total_loss += float(loss.detach().item())
        n_steps    += 1
        loss = loss / grad_accum_steps

        if use_amp:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        if step % grad_accum_steps == 0:
            if use_amp:
                scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            if use_amp:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            if scheduler is not None:
                scheduler.step()

    return total_loss / max(1, n_steps)

## Train: SRL-Aware DirectionalSRL (ARG0 --> ARG1)

### training- fusion mlp

In [ ]:
from typing import List, Dict
from datetime import date
import os
import pickle
import random
import numpy as np
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from datetime import datetime
now = datetime.now()



def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def build_label2id(train_dev: List[UtteranceSample]) -> Dict[str, int]:
    label2id = {}
    for s in train_dev:
        if s.frames is None:
            continue
        for fr in s.frames:
            if fr is None or fr.labels is None:
                continue
            for l in fr.labels:
                if l not in label2id:
                    label2id[l] = len(label2id)
    return label2id

def data_processing_for_loader(
    train_dev: List[UtteranceSample],
    train_sample: List[UtteranceSample],
    dev_sample: List[UtteranceSample],
    test_sample: List[UtteranceSample],
    tokenizer,
    max_length: int = 128
):
    label2id = build_label2id(train_dev)
    id2label = {v: k for k, v in label2id.items()}

    train_ds = SRLDataset(train_sample, tokenizer, label2id, max_length=max_length, debug_print=False)
    dev_ds   = SRLDataset(dev_sample,   tokenizer, label2id, max_length=max_length, debug_print=False)
    test_ds  = SRLDataset(test_sample,  tokenizer, label2id, max_length=max_length, debug_print=False)

    return train_ds, dev_ds, test_ds, label2id, id2label

if __name__ == '__main__':
    set_seed(42)

    bert_name = "bert-base-cased"
    batch_size = 16
    max_length = 256
    num_epochs = 5
    grad_accum_steps = 1
    lr = 1e-5

    tokenizer = AutoTokenizer.from_pretrained(bert_name)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    base_path = "/Enter-your-path/SRL_processed/"
    data_class_train = create_utterance_samples(base_path + "train_formality_2.aligned.jsonl")
    data_class_dev   = create_utterance_samples(base_path + "dev_formality_2.aligned.jsonl")
    data_class_test  = create_utterance_samples(base_path + "test_formality_2.aligned.jsonl")

    print(f"Train: {len(data_class_train)}")
    print(f"Dev:   {len(data_class_dev)}")
    print(f"Test:  {len(data_class_test)}")

    train_dev_data = data_class_train + data_class_dev

    train_ds, dev_ds, test_ds, label2id, id2label = data_processing_for_loader(
        train_dev_data,
        data_class_train,
        data_class_dev,
        data_class_test,
        tokenizer,
        max_length=max_length
    )

    pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    collate = lambda b: srl_collate_ulevel(b, pad_token_id=pad_token_id, pad_label_id=-100)

    use_pin_memory = torch.cuda.is_available()

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate,
        num_workers=2,
        pin_memory=use_pin_memory
    )
    dev_loader = DataLoader(
        dev_ds,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate,
        num_workers=2,
        pin_memory=use_pin_memory
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate,
        num_workers=2,
        pin_memory=use_pin_memory
    )

    model = DirectionalSRL_fusionmlp(
        bert_name=bert_name,
        num_labels=len(label2id),
        use_indicator=True,
        use_distance=True,
        indicator_dim=10,
        lstm_hidden=768,
        mlp_hidden=mlp_layer,
        pos_dim=50,
        max_distance=128,
        dropout=0.1
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    total_steps = len(train_loader) * num_epochs // max(1, grad_accum_steps)
    warmup_steps = int(0.1 * total_steps)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    history = {
        "epoch": [],
        "train_loss": [],
        "dev_loss": [],
        "dev_acc": []
    }

    best_acc = -1.0
    today = date.today()
    model_name = "directionl_fusion_mlp"

    save_dir = "/Enter-your-path/model_log"
    os.makedirs(save_dir, exist_ok=True)

    best_path = os.path.join(
        save_dir,
         now.strftime('%m-%d-%Y_%H-%M-%S') + "_best_directional_formality"+model_name+".ckpt")

    print(f"Training DirectionalSRL for formality for {num_epochs} epochs...")

    for epoch in range(num_epochs):
        tr_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            device=device,
            scheduler=scheduler,
            grad_accum_steps=grad_accum_steps,
            amp=True
        )

        dev_loss, dev_acc = eval_formality(model, dev_loader, device=device)

        history["epoch"].append(epoch + 1)
        history["train_loss"].append(tr_loss)
        history["dev_loss"].append(dev_loss)
        history["dev_acc"].append(dev_acc)

        print(
            f"Epoch {epoch+1:02d}: "
            f"train_loss={tr_loss:.4f}  "
            f"dev_loss={dev_loss:.4f}  "
            f"dev_acc={dev_acc:.4f}"
        )

        if dev_acc > best_acc:
            best_acc = dev_acc
            torch.save({
                "model_state": model.state_dict(),
                "label2id": label2id,
                "id2label": id2label,
                "best_acc": best_acc,
                "architecture": "Directional",
                "hparams": {
                    "bert_name": bert_name,
                    "num_labels": len(label2id),
                    "batch_size": batch_size,
                    "max_length": max_length,
                    "lr": lr,
                    "dropout": 0.1,
                    "use_indicator": True,
                    "use_distance": True,
                    "indicator_dim": 10,
                    "lstm_hidden": 768,
                    "mlp_hidden": 256,
                    "pos_dim": 50,
                    "max_distance": 128,
                }
            }, best_path)
            print(f"New best model saved to: {best_path}")

    hist_dir = os.path.join(save_dir, "loss_history")
    os.makedirs(hist_dir, exist_ok=True)

    hist_file_name = os.path.join(
        hist_dir,
         now.strftime('%m-%d-%Y_%H-%M-%S') + "_directional_formality"+model_name+".pkl"
    )

    with open(hist_file_name, 'wb') as f:
        pickle.dump(history, f)

    print(f"History saved to {hist_file_name}")

    # Reload best checkpoint before final test
    print("Loading best checkpoint for final test evaluation...")
    best_ckpt = torch.load(best_path, map_location=device, weights_only=False)
    model.load_state_dict(best_ckpt["model_state"], strict=False)
    model.eval()

    test_loss, test_acc = eval_formality(model, test_loader, device=device)
    print(f"Best-checkpoint Test loss={test_loss:.4f}  test_acc={test_acc:.4f}")

### training - fusion_nomlp 

In [ ]:
from typing import List, Dict
from datetime import date
import os
import pickle
import random
import numpy as np
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from datetime import datetime
now = datetime.now()



def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def build_label2id(train_dev: List[UtteranceSample]) -> Dict[str, int]:
    label2id = {}
    for s in train_dev:
        if s.frames is None:
            continue
        for fr in s.frames:
            if fr is None or fr.labels is None:
                continue
            for l in fr.labels:
                if l not in label2id:
                    label2id[l] = len(label2id)
    return label2id

def data_processing_for_loader(
    train_dev: List[UtteranceSample],
    train_sample: List[UtteranceSample],
    dev_sample: List[UtteranceSample],
    test_sample: List[UtteranceSample],
    tokenizer,
    max_length: int = 128
):
    label2id = build_label2id(train_dev)
    id2label = {v: k for k, v in label2id.items()}

    train_ds = SRLDataset(train_sample, tokenizer, label2id, max_length=max_length, debug_print=False)
    dev_ds   = SRLDataset(dev_sample,   tokenizer, label2id, max_length=max_length, debug_print=False)
    test_ds  = SRLDataset(test_sample,  tokenizer, label2id, max_length=max_length, debug_print=False)

    return train_ds, dev_ds, test_ds, label2id, id2label

if __name__ == '__main__':
    set_seed(42)

    bert_name = "bert-base-cased"
    batch_size = 16
    max_length = 256
    num_epochs = 5
    grad_accum_steps = 1
    lr = 1e-5

    tokenizer = AutoTokenizer.from_pretrained(bert_name)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    base_path = "/Enter-your-path/SRL_processed/"
    data_class_train = create_utterance_samples(base_path + "train_formality_2.aligned.jsonl")
    data_class_dev   = create_utterance_samples(base_path + "dev_formality_2.aligned.jsonl")
    data_class_test  = create_utterance_samples(base_path + "test_formality_2.aligned.jsonl")

    print(f"Train: {len(data_class_train)}")
    print(f"Dev:   {len(data_class_dev)}")
    print(f"Test:  {len(data_class_test)}")

    train_dev_data = data_class_train + data_class_dev

    train_ds, dev_ds, test_ds, label2id, id2label = data_processing_for_loader(
        train_dev_data,
        data_class_train,
        data_class_dev,
        data_class_test,
        tokenizer,
        max_length=max_length
    )

    pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    collate = lambda b: srl_collate_ulevel(b, pad_token_id=pad_token_id, pad_label_id=-100)

    use_pin_memory = torch.cuda.is_available()

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate,
        num_workers=2,
        pin_memory=use_pin_memory
    )
    dev_loader = DataLoader(
        dev_ds,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate,
        num_workers=2,
        pin_memory=use_pin_memory
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate,
        num_workers=2,
        pin_memory=use_pin_memory
    )

    model = DirectionalSRL_fusionnomlp(
        bert_name=bert_name,
        num_labels=len(label2id),
        use_indicator=True,
        use_distance=True,
        indicator_dim=10,
        lstm_hidden=768,
        mlp_hidden=mlp_layer,
        pos_dim=50,
        max_distance=128,
        dropout=0.1
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    total_steps = len(train_loader) * num_epochs // max(1, grad_accum_steps)
    warmup_steps = int(0.1 * total_steps)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    history = {
        "epoch": [],
        "train_loss": [],
        "dev_loss": [],
        "dev_acc": []
    }

    best_acc = -1.0
    today = date.today()
    model_name = "directionl_fusion_nomlp"

    save_dir = "/Enter-your-path/model_log"
    os.makedirs(save_dir, exist_ok=True)

    best_path = os.path.join(
        save_dir,
         now.strftime('%m-%d-%Y_%H-%M-%S') + "_best_directional_"+model_name+".ckpt")

    print(f"Training DirectionalSRL for formality for {num_epochs} epochs...")

    for epoch in range(num_epochs):
        tr_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            device=device,
            scheduler=scheduler,
            grad_accum_steps=grad_accum_steps,
            amp=True
        )

        dev_loss, dev_acc = eval_formality(model, dev_loader, device=device)

        history["epoch"].append(epoch + 1)
        history["train_loss"].append(tr_loss)
        history["dev_loss"].append(dev_loss)
        history["dev_acc"].append(dev_acc)

        print(
            f"Epoch {epoch+1:02d}: "
            f"train_loss={tr_loss:.4f}  "
            f"dev_loss={dev_loss:.4f}  "
            f"dev_acc={dev_acc:.4f}"
        )

        if dev_acc > best_acc:
            best_acc = dev_acc
            torch.save({
                "model_state": model.state_dict(),
                "label2id": label2id,
                "id2label": id2label,
                "best_acc": best_acc,
                "architecture": "Directional",
                "hparams": {
                    "bert_name": bert_name,
                    "num_labels": len(label2id),
                    "batch_size": batch_size,
                    "max_length": max_length,
                    "lr": lr,
                    "dropout": 0.1,
                    "use_indicator": True,
                    "use_distance": True,
                    "indicator_dim": 10,
                    "lstm_hidden": 768,
                    "mlp_hidden": mlp_layer,
                    "pos_dim": 50,
                    "max_distance": 128,
                }
            }, best_path)
            print(f"New best model saved to: {best_path}")

    hist_dir = os.path.join(save_dir, "loss_history")
    os.makedirs(hist_dir, exist_ok=True)

    hist_file_name = os.path.join(
        hist_dir,
         now.strftime('%m-%d-%Y_%H-%M-%S') + "_directional_formality_"+model_name+".pkl"
    )

    with open(hist_file_name, 'wb') as f:
        pickle.dump(history, f)

    print(f"History saved to {hist_file_name}")

    # Reload best checkpoint before final test
    print("Loading best checkpoint for final test evaluation...")
    best_ckpt = torch.load(best_path, map_location=device, weights_only=False)
    model.load_state_dict(best_ckpt["model_state"], strict=False)
    model.eval()

    test_loss, test_acc = eval_formality(model, test_loader, device=device)
    print(f"Best-checkpoint Test loss={test_loss:.4f}  test_acc={test_acc:.4f}")

### training - last concat mlp

In [ ]:
from typing import List, Dict
from datetime import date, datetime
import os
import pickle
import random
import numpy as np
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def build_label2id(train_dev: List[UtteranceSample]) -> Dict[str, int]:
    label2id = {}
    for s in train_dev:
        if s.frames is None:
            continue
        for fr in s.frames:
            if fr is None or fr.labels is None:
                continue
            for l in fr.labels:
                if l not in label2id:
                    label2id[l] = len(label2id)
    return label2id


def data_processing_for_loader(
    train_dev: List[UtteranceSample],
    train_sample: List[UtteranceSample],
    dev_sample: List[UtteranceSample],
    test_sample: List[UtteranceSample],
    tokenizer,
    max_length: int = 128
):
    label2id = build_label2id(train_dev)
    id2label = {v: k for k, v in label2id.items()}

    train_ds = SRLDataset(train_sample, tokenizer, label2id, max_length=max_length, debug_print=False)
    dev_ds   = SRLDataset(dev_sample,   tokenizer, label2id, max_length=max_length, debug_print=False)
    test_ds  = SRLDataset(test_sample,  tokenizer, label2id, max_length=max_length, debug_print=False)

    return train_ds, dev_ds, test_ds, label2id, id2label


if __name__ == '__main__':
    set_seed(42)
    now = datetime.now()

    bert_name = "bert-base-cased"
    batch_size = 16
    max_length = 256
    num_epochs = 5
    grad_accum_steps = 1
    lr = 1e-5
    mlp_layer = 300

    tokenizer = AutoTokenizer.from_pretrained(bert_name)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    base_path = "/Enter-your-path/SRL_processed/"
    data_class_train = create_utterance_samples(base_path + "train_formality_2.aligned.jsonl")
    data_class_dev   = create_utterance_samples(base_path + "dev_formality_2.aligned.jsonl")
    data_class_test  = create_utterance_samples(base_path + "test_formality_2.aligned.jsonl")

    print(f"Train: {len(data_class_train)}")
    print(f"Dev:   {len(data_class_dev)}")
    print(f"Test:  {len(data_class_test)}")

    train_dev_data = data_class_train + data_class_dev

    train_ds, dev_ds, test_ds, label2id, id2label = data_processing_for_loader(
        train_dev_data,
        data_class_train,
        data_class_dev,
        data_class_test,
        tokenizer,
        max_length=max_length
    )

    pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    collate = lambda b: srl_collate_ulevel(b, pad_token_id=pad_token_id, pad_label_id=-100)

    use_pin_memory = torch.cuda.is_available()

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate,
        num_workers=2,
        pin_memory=use_pin_memory
    )
    dev_loader = DataLoader(
        dev_ds,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate,
        num_workers=2,
        pin_memory=use_pin_memory
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate,
        num_workers=2,
        pin_memory=use_pin_memory
    )

    # --------------------------------------------------
    # last-layer concatenation + MLP
    # --------------------------------------------------
    model = DirectionalSRL_lastlayermlp(
        bert_name=bert_name,
        num_labels=len(label2id),
        use_indicator=True,
        use_distance=True,
        indicator_dim=10,
        lstm_hidden=768,
        mlp_hidden=mlp_layer,
        pos_dim=50,
        max_distance=128,
        dropout=0.1
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    total_steps = len(train_loader) * num_epochs // max(1, grad_accum_steps)
    warmup_steps = int(0.1 * total_steps)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    history = {"epoch": [], "train_loss": [], "dev_loss": [], "dev_acc": []}
    best_acc = -1.0

    save_dir = "/Enter-your-path/model_log"
    os.makedirs(save_dir, exist_ok=True)

    model_name = "directional_lastlayer_mlp"
    best_path = os.path.join(
        save_dir,
        now.strftime('%m-%d-%Y_%H-%M-%S') + "_best_directional_" + model_name + ".ckpt"
    )

    print(f"Training {model_name} for {num_epochs} epochs...")

    for epoch in range(num_epochs):
        tr_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            device=device,
            scheduler=scheduler,
            grad_accum_steps=grad_accum_steps,
            amp=True
        )

        dev_loss, dev_acc = eval_formality(model, dev_loader, device=device)

        history["epoch"].append(epoch + 1)
        history["train_loss"].append(tr_loss)
        history["dev_loss"].append(dev_loss)
        history["dev_acc"].append(dev_acc)

        print(
            f"Epoch {epoch+1:02d}: "
            f"train_loss={tr_loss:.4f}  "
            f"dev_loss={dev_loss:.4f}  "
            f"dev_acc={dev_acc:.4f}"
        )

        if dev_acc > best_acc:
            best_acc = dev_acc
            torch.save({
                "model_state": model.state_dict(),
                "label2id": label2id,
                "id2label": id2label,
                "best_acc": best_acc,
                "architecture": "Directional_LastLayer_MLP",
                "hparams": {
                    "bert_name": bert_name,
                    "num_labels": len(label2id),
                    "batch_size": batch_size,
                    "max_length": max_length,
                    "lr": lr,
                    "dropout": 0.1,
                    "use_indicator": True,
                    "use_distance": True,
                    "indicator_dim": 10,
                    "lstm_hidden": 768,
                    "mlp_hidden": mlp_layer,
                    "pos_dim": 50,
                    "max_distance": 128,
                }
            }, best_path)
            print(f"New best model saved to: {best_path}")

    hist_dir = os.path.join(save_dir, "loss_history")
    os.makedirs(hist_dir, exist_ok=True)

    hist_file_name = os.path.join(
        hist_dir,
        now.strftime('%m-%d-%Y_%H-%M-%S') + "_directional_formality_" + model_name + ".pkl"
    )

    with open(hist_file_name, 'wb') as f:
        pickle.dump(history, f)

    print(f"History saved to {hist_file_name}")

    print("Loading best checkpoint for final test evaluation...")
    best_ckpt = torch.load(best_path, map_location=device, weights_only=False)
    model.load_state_dict(best_ckpt["model_state"], strict=True)
    model.eval()

    test_loss, test_acc = eval_formality(model, test_loader, device=device)
    print(f"Best-checkpoint Test loss={test_loss:.4f}  test_acc={test_acc:.4f}")

### training - last concat no mlp

In [ ]:
from typing import List, Dict
from datetime import date, datetime
import os
import pickle
import random
import numpy as np
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def build_label2id(train_dev: List[UtteranceSample]) -> Dict[str, int]:
    label2id = {}
    for s in train_dev:
        if s.frames is None:
            continue
        for fr in s.frames:
            if fr is None or fr.labels is None:
                continue
            for l in fr.labels:
                if l not in label2id:
                    label2id[l] = len(label2id)
    return label2id


def data_processing_for_loader(
    train_dev: List[UtteranceSample],
    train_sample: List[UtteranceSample],
    dev_sample: List[UtteranceSample],
    test_sample: List[UtteranceSample],
    tokenizer,
    max_length: int = 128
):
    label2id = build_label2id(train_dev)
    id2label = {v: k for k, v in label2id.items()}

    train_ds = SRLDataset(train_sample, tokenizer, label2id, max_length=max_length, debug_print=False)
    dev_ds   = SRLDataset(dev_sample,   tokenizer, label2id, max_length=max_length, debug_print=False)
    test_ds  = SRLDataset(test_sample,  tokenizer, label2id, max_length=max_length, debug_print=False)

    return train_ds, dev_ds, test_ds, label2id, id2label


if __name__ == '__main__':
    set_seed(42)
    now = datetime.now()

    bert_name = "bert-base-cased"
    batch_size = 16
    max_length = 256
    num_epochs = 5
    grad_accum_steps = 1
    lr = 1e-5

    tokenizer = AutoTokenizer.from_pretrained(bert_name)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    base_path = "/Enter-your-path/SRL_processed/"
    data_class_train = create_utterance_samples(base_path + "train_formality_2.aligned.jsonl")
    data_class_dev   = create_utterance_samples(base_path + "dev_formality_2.aligned.jsonl")
    data_class_test  = create_utterance_samples(base_path + "test_formality_2.aligned.jsonl")

    print(f"Train: {len(data_class_train)}")
    print(f"Dev:   {len(data_class_dev)}")
    print(f"Test:  {len(data_class_test)}")

    train_dev_data = data_class_train + data_class_dev

    train_ds, dev_ds, test_ds, label2id, id2label = data_processing_for_loader(
        train_dev_data,
        data_class_train,
        data_class_dev,
        data_class_test,
        tokenizer,
        max_length=max_length
    )

    pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    collate = lambda b: srl_collate_ulevel(b, pad_token_id=pad_token_id, pad_label_id=-100)

    use_pin_memory = torch.cuda.is_available()

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate,
        num_workers=2,
        pin_memory=use_pin_memory
    )
    dev_loader = DataLoader(
        dev_ds,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate,
        num_workers=2,
        pin_memory=use_pin_memory
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate,
        num_workers=2,
        pin_memory=use_pin_memory
    )

    # --------------------------------------------------
    # last-layer concatenation + no MLP
    # --------------------------------------------------
    model = DirectionalSRL_lastlayernomlp(
        bert_name=bert_name,
        num_labels=len(label2id),
        use_indicator=True,
        use_distance=True,
        indicator_dim=10,
        lstm_hidden=768,
        pos_dim=50,
        max_distance=128,
        dropout=0.1
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    total_steps = len(train_loader) * num_epochs // max(1, grad_accum_steps)
    warmup_steps = int(0.1 * total_steps)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    history = {"epoch": [], "train_loss": [], "dev_loss": [], "dev_acc": []}
    best_acc = -1.0

    save_dir = "/Enter-your-path/model_log"
    os.makedirs(save_dir, exist_ok=True)

    model_name = "directional_lastlayer_nomlp"
    best_path = os.path.join(
        save_dir,
        now.strftime('%m-%d-%Y_%H-%M-%S') + "_best_directional_" + model_name + ".ckpt"
    )

    print(f"Training {model_name} for {num_epochs} epochs...")

    for epoch in range(num_epochs):
        tr_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            device=device,
            scheduler=scheduler,
            grad_accum_steps=grad_accum_steps,
            amp=True
        )

        dev_loss, dev_acc = eval_formality(model, dev_loader, device=device)

        history["epoch"].append(epoch + 1)
        history["train_loss"].append(tr_loss)
        history["dev_loss"].append(dev_loss)
        history["dev_acc"].append(dev_acc)

        print(
            f"Epoch {epoch+1:02d}: "
            f"train_loss={tr_loss:.4f}  "
            f"dev_loss={dev_loss:.4f}  "
            f"dev_acc={dev_acc:.4f}"
        )

        if dev_acc > best_acc:
            best_acc = dev_acc
            torch.save({
                "model_state": model.state_dict(),
                "label2id": label2id,
                "id2label": id2label,
                "best_acc": best_acc,
                "architecture": "Directional_LastLayer_NoMLP",
                "hparams": {
                    "bert_name": bert_name,
                    "num_labels": len(label2id),
                    "batch_size": batch_size,
                    "max_length": max_length,
                    "lr": lr,
                    "dropout": 0.1,
                    "use_indicator": True,
                    "use_distance": True,
                    "indicator_dim": 10,
                    "lstm_hidden": 768,
                    "pos_dim": 50,
                    "max_distance": 128,
                }
            }, best_path)
            print(f"New best model saved to: {best_path}")

    hist_dir = os.path.join(save_dir, "loss_history")
    os.makedirs(hist_dir, exist_ok=True)

    hist_file_name = os.path.join(
        hist_dir,
        now.strftime('%m-%d-%Y_%H-%M-%S') + "_directional_formality_" + model_name + ".pkl"
    )

    with open(hist_file_name, 'wb') as f:
        pickle.dump(history, f)

    print(f"History saved to {hist_file_name}")

    print("Loading best checkpoint for final test evaluation...")
    best_ckpt = torch.load(best_path, map_location=device, weights_only=False)
    model.load_state_dict(best_ckpt["model_state"], strict=True)
    model.eval()

    test_loss, test_acc = eval_formality(model, test_loader, device=device)
    print(f"Best-checkpoint Test loss={test_loss:.4f}  test_acc={test_acc:.4f}")

# 5. Vanilla BERT Formality Model

## Dataset (no SRL)

In [ ]:
class UtteranceOnlyDataset(Dataset):
    """Tokenises the full utterance as a plain string. No SRL features."""
    def __init__(self, samples, tokenizer, max_length=256):
        self.samples    = samples
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        utt  = self.samples[idx]
        text = " ".join(utt.words)
        enc  = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding=False,
            return_tensors=None
        )
        res = {
            "input_ids":      torch.tensor(enc["input_ids"],      dtype=torch.long),
            "attention_mask": torch.tensor(enc["attention_mask"], dtype=torch.long),
        }
        if "token_type_ids" in enc:
            res["token_type_ids"] = torch.tensor(enc["token_type_ids"], dtype=torch.long)
        if utt.formality is not None:
            res["formality"] = torch.tensor(float(utt.formality), dtype=torch.float)
        return res

## Collate (Vanilla)

In [ ]:
def utterance_collate(batch, pad_token_id: int):
    B     = len(batch)
    max_L = max(x["input_ids"].size(0) for x in batch)

    input_ids      = torch.full((B, max_L), pad_token_id, dtype=torch.long)
    attention_mask = torch.zeros((B, max_L), dtype=torch.long)

    has_token_type = "token_type_ids" in batch[0]
    if has_token_type:
        token_type_ids = torch.zeros((B, max_L), dtype=torch.long)

    has_formality = "formality" in batch[0]
    if has_formality:
        formality = torch.zeros((B,), dtype=torch.float)

    for i, item in enumerate(batch):
        L = item["input_ids"].size(0)
        input_ids[i, :L]      = item["input_ids"]
        attention_mask[i, :L] = item["attention_mask"]
        if has_token_type:
            token_type_ids[i, :L] = item["token_type_ids"]
        if has_formality:
            formality[i] = item["formality"]

    res = {"input_ids": input_ids, "attention_mask": attention_mask}
    if has_token_type:
        res["token_type_ids"] = token_type_ids
    if has_formality:
        res["formality"] = formality
    return res

In [ ]:
def data_processing_for_loader_vanilla(
    train_sample: List[UtteranceSample],
    dev_sample:   List[UtteranceSample],
    test_sample:  List[UtteranceSample],
    tokenizer,
    max_length: int = 256
):
    train_ds = UtteranceOnlyDataset(train_sample, tokenizer, max_length=max_length)
    dev_ds   = UtteranceOnlyDataset(dev_sample,   tokenizer, max_length=max_length)
    test_ds  = UtteranceOnlyDataset(test_sample,  tokenizer, max_length=max_length)
    return train_ds, dev_ds, test_ds, None, None

## Vanilla BERT Model

In [ ]:
mlp_layer = 300

In [ ]:
from transformers import AutoModel, AutoConfig


class VanillaBertFormality(nn.Module):
    """Baseline BERT for formality regression. Uses [CLS] pooled output + MLP head."""

    def __init__(self, bert_name="bert-base-uncased", mlp_hidden=mlp_layer, dropout=0.1):
        super().__init__()
        self.config  = AutoConfig.from_pretrained(bert_name)
        self.bert    = AutoModel.from_pretrained(bert_name)
        H            = self.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.reg_head = nn.Sequential(
            nn.Linear(H, mlp_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 1)
        )

    def forward(self, input_ids, attention_mask, token_type_ids=None,
                formality=None, **kwargs):
        out = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        pooled = out.pooler_output if (hasattr(out, "pooler_output") and out.pooler_output is not None) \
                 else out.last_hidden_state[:, 0, :]
        pooled = self.dropout(pooled)
        score  = self.reg_head(pooled).squeeze(-1)

        loss = torch.tensor(0.0, device=score.device)
        if formality is not None:
            loss = nn.BCEWithLogitsLoss()(score, formality)
        return score, loss, None


#Updated without additional mlphead= 256 

class VanillaBertFormality_nomlp(nn.Module):
    """Baseline BERT for formality prediction using pooled [CLS] + linear head."""

    def __init__(self, bert_name="bert-base-uncased", dropout=0.1):
        super().__init__()
        self.config = AutoConfig.from_pretrained(bert_name)
        self.bert = AutoModel.from_pretrained(bert_name)
        H = self.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.reg_head = nn.Linear(H, 1)

    def forward(self, input_ids, attention_mask, token_type_ids=None,
                formality=None, **kwargs):
        out = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        pooled = out.pooler_output if (
            hasattr(out, "pooler_output") and out.pooler_output is not None
        ) else out.last_hidden_state[:, 0, :]

        score = self.reg_head(self.dropout(pooled)).squeeze(-1)

        loss = torch.tensor(0.0, device=score.device)
        if formality is not None:
            loss = nn.BCEWithLogitsLoss()(score, formality.float())

        return score, loss, None


## Train: Vanilla BERT- with mlp 

In [ ]:
if __name__ == '__main__':
    bert_name = "bert-base-cased"
    tokenizer = AutoTokenizer.from_pretrained(bert_name)
    device    = "cuda" if torch.cuda.is_available() else "cpu"

    # base_path = "/Enter-your-path/Final_dataset/"
    
    train_df = load_pkl("/Enter-your-path/all_train.pkl")
    dev_df = load_pkl("/Enter-your-path/all_dev.pkl")
    test_df = load_pkl("/Enter-your-path/all_test.pkl")

    data_class_train = create_formality_samples_from_df(train_df)
    data_class_dev   = create_formality_samples_from_df(dev_df)
    data_class_test  = create_formality_samples_from_df(test_df)

    print(f"Train: {len(data_class_train)}  Dev: {len(data_class_dev)}  Test: {len(data_class_test)}")

    train_ds, dev_ds, test_ds, _, _ = data_processing_for_loader_vanilla(
        data_class_train, data_class_dev, data_class_test, tokenizer, max_length=256
    )

    pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    collate      = lambda b: utterance_collate(b, pad_token_id=pad_token_id)

    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  collate_fn=collate)
    dev_loader   = DataLoader(dev_ds,   batch_size=16, shuffle=False, collate_fn=collate)

    model = VanillaBertFormality(bert_name=bert_name, mlp_hidden=mlp_layer, dropout=0.1).to(device) #before fusion#
    # model = VanillaBertFormality(bert_name=bert_name, dropout=0.1).to(device)

    num_epochs       = 5
    grad_accum_steps = 1
    optimizer        = torch.optim.AdamW(model.parameters(), lr=1e-5)
    total_steps      = len(train_loader) * num_epochs // max(1, grad_accum_steps)
    warmup_steps     = int(0.1 * total_steps)
    scheduler        = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
    )

    history   = {"epoch": [], "train_loss": [], "dev_loss": [], "dev_acc": []}
    best_corr = -1.0
    now = datetime.now()
    today = now.strftime("%Y-%m-%d_%H-%M-%S")
    model_name = "vanilla-bert-cased-with-mlp"
    
    # best_path = os.path.join(
    #     save_dir,
    #      now.strftime('%m-%d-%Y_%H-%M-%S') + "_best_directional_formality_nomlp.ckpt")
    
    save_dir  = "/Enter-your-path/model_log"
    best_path = os.path.join(save_dir, today+"_epochs_"+str(num_epochs)+"_best_model_"+model_name+"_formality.ckpt")

    print(f"Training VanillaBertFormality for {num_epochs} epochs...")
    for epoch in range(num_epochs):
        tr_loss            = train_one_epoch(model, train_loader, optimizer, device=device,
                                             scheduler=scheduler, grad_accum_steps=grad_accum_steps, amp=True)
        dev_loss, dev_acc = eval_formality(model, dev_loader, device=device)

        history["epoch"].append(epoch + 1)
        history["train_loss"].append(tr_loss)
        history["dev_loss"].append(dev_loss)
        history["dev_acc"].append(dev_acc)
        print(f"Epoch {epoch+1}: train_loss={tr_loss:.4f}  dev_loss={dev_loss:.4f}  dev_acc={dev_acc:.4f}")

        if dev_acc > best_corr:
            best_corr = dev_acc
            torch.save({
                "model_state": model.state_dict(),
                "label2id":    None,
                "best_corr":   best_corr,
                "hparams":     {"bert_name": bert_name},
            }, best_path)
            print("new best model saved")
    

            
    hist_dir  = os.path.join(save_dir, "loss_history")
    hist_file = os.path.join(hist_dir,  today +"_"+model_name+"_formality.pkl")
    with open(hist_file, 'wb') as f:
        pickle.dump(history, f)
    print(f"History saved to {hist_file}")

## vanilla -no mlp

In [ ]:
if __name__ == '__main__':
    bert_name = "bert-base-cased"
    tokenizer = AutoTokenizer.from_pretrained(bert_name)
    device    = "cuda" if torch.cuda.is_available() else "cpu"

    # base_path = "/Enter-your-path/Final_dataset/"
    
    train_df = load_pkl("/Enter-your-path/all_train.pkl")
    dev_df = load_pkl("/Enter-your-path/all_dev.pkl")
    test_df = load_pkl("/Enter-your-path/all_test.pkl")

    data_class_train = create_formality_samples_from_df(train_df)
    data_class_dev   = create_formality_samples_from_df(dev_df)
    data_class_test  = create_formality_samples_from_df(test_df)

    print(f"Train: {len(data_class_train)}  Dev: {len(data_class_dev)}  Test: {len(data_class_test)}")

    train_ds, dev_ds, test_ds, _, _ = data_processing_for_loader_vanilla(
        data_class_train, data_class_dev, data_class_test, tokenizer, max_length=256
    )

    pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    collate      = lambda b: utterance_collate(b, pad_token_id=pad_token_id)

    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  collate_fn=collate)
    dev_loader   = DataLoader(dev_ds,   batch_size=16, shuffle=False, collate_fn=collate)

    # model = VanillaBertFormality(bert_name=bert_name, mlp_hidden=mlp_layer, dropout=0.1).to(device) #before fusion#
    model = VanillaBertFormality_nomlp(bert_name=bert_name, dropout=0.1).to(device)

    num_epochs       = 5
    grad_accum_steps = 1
    optimizer        = torch.optim.AdamW(model.parameters(), lr=1e-5)
    total_steps      = len(train_loader) * num_epochs // max(1, grad_accum_steps)
    warmup_steps     = int(0.1 * total_steps)
    scheduler        = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
    )

    history   = {"epoch": [], "train_loss": [], "dev_loss": [], "dev_acc": []}
    best_corr = -1.0
    now = datetime.now()
    today = now.strftime("%Y-%m-%d_%H-%M-%S")
    model_name = "vanilla-bert-cased-with-nomlp"
    
    # best_path = os.path.join(
    #     save_dir,
    #      now.strftime('%m-%d-%Y_%H-%M-%S') + "_best_directional_formality_nomlp.ckpt")
    
    save_dir  = "/Enter-your-path/model_log"
    best_path = os.path.join(save_dir, today+"_epochs_"+str(num_epochs)+"_best_model_"+model_name+"_formality.ckpt")

    print(f"Training VanillaBertFormality for {num_epochs} epochs...")
    for epoch in range(num_epochs):
        tr_loss            = train_one_epoch(model, train_loader, optimizer, device=device,
                                             scheduler=scheduler, grad_accum_steps=grad_accum_steps, amp=True)
        dev_loss, dev_acc = eval_formality(model, dev_loader, device=device)

        history["epoch"].append(epoch + 1)
        history["train_loss"].append(tr_loss)
        history["dev_loss"].append(dev_loss)
        history["dev_acc"].append(dev_acc)
        print(f"Epoch {epoch+1}: train_loss={tr_loss:.4f}  dev_loss={dev_loss:.4f}  dev_acc={dev_acc:.4f}")

        if dev_acc > best_corr:
            best_corr = dev_acc
            torch.save({
                "model_state": model.state_dict(),
                "label2id":    None,
                "best_corr":   best_corr,
                "hparams":     {"bert_name": bert_name},
            }, best_path)
            print("new best model saved")
    

            
    hist_dir  = os.path.join(save_dir, "loss_history")
    hist_file = os.path.join(hist_dir,  today + "_"+model_name+"_formality.pkl")
    with open(hist_file, 'wb') as f:
        pickle.dump(history, f)
    print(f"History saved to {hist_file}")

# 6. Testing (Utterance Level)

### Testing- final

In [ ]:
import os
import numpy as np
import torch
import pandas as pd
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from datetime import datetime

now = datetime.now()

mlp_layer = 300


# ------------------------------------------------------------
# 0) Helpers
# ------------------------------------------------------------
def is_vanilla_arch(model_arch: str) -> bool:
    return model_arch in {"Vanilla_MLP", "Vanilla_NoMLP"}


@torch.no_grad()
def test_formality(model, dataloader, device="cuda", debug=False):
    model.eval()
    all_preds, all_golds = [], []
    total_loss, n_batches = 0.0, 0

    for batch_idx, batch in enumerate(dataloader):
        batch = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}

        preds, loss, _ = model(**batch)
        total_loss += float(loss.item())
        n_batches += 1

        if "formality" in batch:
            all_preds.append(preds.detach().cpu().numpy())
            all_golds.append(batch["formality"].detach().cpu().numpy())

        if debug and batch_idx == 0:
            print("Batch keys:", list(batch.keys()))
            print("Pred shape:", tuple(preds.shape))

    avg_loss = total_loss / max(1, n_batches)

    if len(all_preds) == 0:
        return avg_loss, 0.0, None, None, None

    logits = np.concatenate(all_preds, axis=0)
    golds = np.concatenate(all_golds, axis=0)

    probs = torch.sigmoid(torch.tensor(logits)).numpy()
    binary = (probs >= 0.5).astype(float)
    acc = (binary == golds).mean()

    if debug:
        print("logits min/max:", float(np.min(logits)), float(np.max(logits)))
        print("probs  min/max:", float(np.min(probs)), float(np.max(probs)))
        print("pred positive rate:", float(np.mean(binary)))
        print("gold positive rate:", float(np.mean(golds)))
        print("first 10 probs:", probs[:10])
        print("first 10 binary:", binary[:10])
        print("first 10 golds:", golds[:10])

    return avg_loss, acc, probs, binary, golds


# ------------------------------------------------------------
# 1) Build correct model from checkpoint metadata / experiment spec
# ------------------------------------------------------------
def load_model_and_labels(exp, device):
    print(f"Loading {exp['name']}...")
    ckpt_path = exp["ckpt_file"]
    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)

    model_arch = exp["architecture"]
    bert_name = exp.get("bert_name", "bert-base-cased")
    label2id = checkpoint.get("label2id", None)

    if model_arch == "Vanilla_MLP":
        model = VanillaBertFormality(
            bert_name=bert_name,
            mlp_hidden=exp.get("mlp_hidden", mlp_layer),
            dropout=0.0
        )
        label2id = None

    elif model_arch == "Vanilla_NoMLP":
        model = VanillaBertFormality_nomlp(
            bert_name=bert_name,
            dropout=0.0
        )
        label2id = None

    elif model_arch == "SRL_Fusion_MLP":
        if label2id is None:
            raise ValueError("SRL model requires label2id in checkpoint.")
        model = DirectionalSRL_fusionmlp(
            bert_name=bert_name,
            num_labels=len(label2id),
            use_indicator=True,
            use_distance=True,
            indicator_dim=10,
            lstm_hidden=768,
            mlp_hidden=exp.get("mlp_hidden", mlp_layer),
            pos_dim=50,
            max_distance=128,
            dropout=0.0
        )

    elif model_arch == "SRL_Fusion_NoMLP":
        if label2id is None:
            raise ValueError("SRL model requires label2id in checkpoint.")
        model = DirectionalSRL_fusionnomlp(
            bert_name=bert_name,
            num_labels=len(label2id),
            use_indicator=True,
            use_distance=True,
            indicator_dim=10,
            lstm_hidden=768,
            mlp_hidden=exp.get("mlp_hidden", mlp_layer),  # harmless if unused
            pos_dim=50,
            max_distance=128,
            dropout=0.0
        )

    elif model_arch == "SRL_LastLayer_MLP":
        if label2id is None:
            raise ValueError("SRL model requires label2id in checkpoint.")
        model = DirectionalSRL_lastlayermlp(
            bert_name=bert_name,
            num_labels=len(label2id),
            use_indicator=True,
            use_distance=True,
            indicator_dim=10,
            lstm_hidden=768,
            mlp_hidden=exp.get("mlp_hidden", mlp_layer),
            pos_dim=50,
            max_distance=128,
            dropout=0.0
        )

    elif model_arch == "SRL_LastLayer_NoMLP":
        if label2id is None:
            raise ValueError("SRL model requires label2id in checkpoint.")
        model = DirectionalSRL_lastlayernomlp(
            bert_name=bert_name,
            num_labels=len(label2id),
            use_indicator=True,
            use_distance=True,
            indicator_dim=10,
            lstm_hidden=768,
            mlp_hidden=exp.get("mlp_hidden", mlp_layer),  # harmless if unused
            pos_dim=50,
            max_distance=128,
            dropout=0.0
        )

    else:
        raise ValueError(f"Unknown architecture: {model_arch}")

    model.load_state_dict(checkpoint["model_state"], strict=True)
    model.to(device)
    model.eval()
    return model, label2id


# ------------------------------------------------------------
# 2) Build the right dataloader for each model
# ------------------------------------------------------------
def build_test_dataloader(exp, data_class_test, tokenizer, label2id=None, batch_size=16, max_length=256):
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id

    if is_vanilla_arch(exp["architecture"]):
        test_ds = UtteranceOnlyDataset(data_class_test, tokenizer, max_length=max_length)
        collate = lambda b: utterance_collate(b, pad_token_id=pad_id)
        test_dl = DataLoader(
            test_ds,
            batch_size=batch_size,
            shuffle=False,
            collate_fn=collate
        )
    else:
        if label2id is None:
            raise ValueError("SRL dataloader requires label2id, but got None.")
        test_ds = SRLDataset(data_class_test, tokenizer, label2id, max_length=max_length, debug_print=False)
        collate = lambda b: srl_collate_ulevel(b, pad_token_id=pad_id, pad_label_id=-100)
        test_dl = DataLoader(
            test_ds,
            batch_size=batch_size,
            shuffle=False,
            collate_fn=collate
        )

    return test_ds, test_dl


# ------------------------------------------------------------
# 3) Single experiment runner
# ------------------------------------------------------------
def run_one_test_experiment(exp, data_class_test, device="cuda", batch_size=16, max_length=256, save_dir=None, debug=False):
    bert_name = exp.get("bert_name", "bert-base-cased")
    tokenizer = AutoTokenizer.from_pretrained(bert_name)

    model, label2id = load_model_and_labels(exp, device)
    test_ds, test_dl = build_test_dataloader(
        exp,
        data_class_test,
        tokenizer,
        label2id=label2id,
        batch_size=batch_size,
        max_length=max_length
    )

    avg_loss, acc, probs, binary, golds = test_formality(model, test_dl, device=device, debug=debug)
    print(f"[{exp['name']}] test_loss={avg_loss:.4f}  test_acc={acc:.4f}")

    df = pd.DataFrame({
        "idx": list(range(len(data_class_test))),
        "text": [" ".join(s.words) for s in data_class_test],
        "gold_formality": [float(s.formality) for s in data_class_test],
        "pred_prob": probs,
        "pred_binary": binary,
    })

    if save_dir is not None:
        os.makedirs(save_dir, exist_ok=True)
        save_path = os.path.join(
            save_dir,
            f"{now.strftime('%m-%d-%Y_%H-%M-%S')}_{exp['name']}_test_preds.csv"
        )
        df.to_csv(save_path, index=False)
        print(f"Saved predictions to: {save_path}")

    return {
        "name": exp["name"],
        "architecture": exp["architecture"],
        "test_loss": avg_loss,
        "test_acc": acc,
        "df": df,
    }


# ------------------------------------------------------------
# 4) Main: all 6 experiments
# ------------------------------------------------------------
if __name__ == '__main__':
    device = "cuda" if torch.cuda.is_available() else "cpu"

    test_df = load_pkl("/Enter-your-path/all_test.pkl")
    data_class_test = create_formality_samples_from_df(test_df)

    print(f"Test: {len(data_class_test)} samples")

    EXPERIMENTS = [
        {
            "name": "Vanilla_BERT_MLP",
            "architecture": "Vanilla_MLP",
            "bert_name": "bert-base-cased",
            "mlp_hidden": mlp_layer,
            "ckpt_file": "/Enter-your-path/2026-04-25_14-58-23_epochs_5_best_model_vanilla-bert-cased-with-mlp_formality.ckpt",
        },
        {
            "name": "Vanilla_BERT_NoMLP",
            "architecture": "Vanilla_NoMLP",
            "bert_name": "bert-base-cased",
            "ckpt_file": "/Enter-your-path/2026-04-25_15-17-56_epochs_5_best_model_vanilla-bert-cased-with-nomlp_formality.ckpt",
        },
        {
            "name": "SRLDirectional_Fusion_MLP",
            "architecture": "SRL_Fusion_MLP",
            "bert_name": "bert-base-cased",
            "mlp_hidden": mlp_layer,
            "ckpt_file": "/Enter-your-path/04-25-2026_15-43-19_best_directional_formalitydirectionl_fusion_mlp.ckpt",
        },
        {
            "name": "SRLDirectional_Fusion_NoMLP",
            "architecture": "SRL_Fusion_NoMLP",
            "bert_name": "bert-base-cased",
            "ckpt_file": "/Enter-your-path/04-25-2026_16-10-12_best_directional_directionl_fusion_nomlp.ckpt",
        },
        {
            "name": "SRLDirectional_LastLayer_MLP",
            "architecture": "SRL_LastLayer_MLP",
            "bert_name": "bert-base-cased",
            "mlp_hidden": mlp_layer,
            "ckpt_file": "/Enter-your-path/04-25-2026_16-40-26_best_directional_directional_lastlayer_mlp.ckpt",
        },
        {
            "name": "SRLDirectional_LastLayer_NoMLP",
            "architecture": "SRL_LastLayer_NoMLP",
            "bert_name": "bert-base-cased",
            "ckpt_file": "/Enter-your-path/04-25-2026_17-06-57_best_directional_directional_lastlayer_nomlp.ckpt",
        },
    ]

    results = []
    save_dir = "/Enter-your-path/test_results"

    for i, exp in enumerate(EXPERIMENTS):
        if not os.path.exists(exp["ckpt_file"]):
            print(f"[SKIP] Checkpoint not found: {exp['ckpt_file']}")
            continue

        debug_flag = (i == 0)  # set True only for first run; change if needed

        out = run_one_test_experiment(
            exp,
            data_class_test=data_class_test,
            device=device,
            batch_size=16,
            max_length=256,
            save_dir=save_dir,
            debug=debug_flag
        )
        results.append(out)

    if results:
        summary_df = pd.DataFrame([
            {
                "name": r["name"],
                "architecture": r["architecture"],
                "test_loss": r["test_loss"],
                "test_acc": r["test_acc"],
            }
            for r in results
        ])

        print("\n===== Summary =====")
        print(summary_df.to_string(index=False))

        summary_path = os.path.join(
            save_dir,
            f"{now.strftime('%m-%d-%Y_%H-%M-%S')}_formality_test_summary.csv"
        )
        summary_df.to_csv(summary_path, index=False)
        print(f"Saved summary to: {summary_path}")